# Microproyecto 3: Generación condicional de código con LLMs

**Curso:** Modelos Avanzados para el Procesamiento de Lenguaje Natural
**Integrante:** Jean Pierre Betancourt Padilla
**Fecha:** noviembre de 2026

---

## 1. Portada e introducción

### Resumen

En este microproyecto ajustamos dos modelos de lenguaje de familias distintas para generar
funciones en Python a partir de un enunciado en lenguaje natural. Trabajamos con MBPP, un corpus
de problemas básicos de programación que trae, para cada problema, el enunciado, una solución de
referencia y los `assert` con los que se verifica.

Comparamos cuatro configuraciones para medir cuánto aporta el finetuning con LoRA y qué familia
de modelo resuelve mejor la tarea:

| # | Configuración | Modelo | Entrenamiento |
| :-: | :--- | :--- | :--- |
| 1 | Qwen base | Qwen2.5-1.5B | Sin entrenar (zero-shot) |
| 2 | Qwen + LoRA | Qwen2.5-1.5B | LoRA sobre MBPP `train` |
| 3 | SmolLM2 base | SmolLM2-1.7B | Sin entrenar (zero-shot) |
| 4 | SmolLM2 + LoRA | SmolLM2-1.7B | LoRA sobre MBPP `train` |

### Datos

Usamos `google-research-datasets/mbpp` en su configuración `full` con las particiones oficiales:
`train` (~374 problemas) para el finetuning, `validation` (~90) para vigilar el entrenamiento y
`test` (500) reservado solo para la evaluación final.

El prompt que ve el modelo tiene el enunciado del problema y el nombre con la firma de la función
que hay que escribir. No incluye la solución de referencia, ni las clases y helpers que trae el
código, ni los casos de prueba: así el modelo tiene que resolver el problema y no puede copiarse.
Lo que aprende a generar es el código de referencia completo, con sus imports y sus helpers.

### Evaluación

La métrica es **pass@1**: un problema cuenta como correcto solo si el código generado pasa todos
sus `assert`. Evaluamos los 500 problemas de `test` con una sola solución por problema y
decodificación greedy, y usamos el mismo prompt y el mismo `max_new_tokens` en la versión base y
en la finetuneada de cada modelo.

El código generado se ejecuta en un subproceso aparte con un tiempo límite, nunca dentro del
notebook, porque el modelo puede equivocarse y colgar la sesión.

### Entorno

El notebook se escribió en WSL y se ejecuta completo en Google Colab con GPU L4 o A100. En L4 y
A100 usamos bf16; si la sesión cae en una T4, pasamos a fp16 con `GradScaler` sin cambiar código.
Los adaptadores LoRA y los resultados quedan en Google Drive para sobrevivir a una desconexión.

Todo el pipeline es reproducible: fijamos la semilla, la revisión del dataset y las de los dos
modelos, de modo que dos ejecuciones parten de los mismos datos y de los mismos pesos. El notebook
sigue las once secciones que pide el enunciado y se puede ejecutar de principio a fin con
`Run All`.


## 2. Librerías

Dejamos listo el entorno de ejecución antes de tocar los datos: instalamos las librerías,
verificamos la GPU y la precisión mixta, montamos Drive y fijamos la semilla.


Instalamos las versiones de Hugging Face que usamos en el proyecto. Colab trae `torchao` 0.10 y
PEFT 0.21 exige 0.16 o superior; como no usamos cuantización con torchao, la desinstalamos para que
PEFT no la detecte al cargar el modelo.


In [ ]:
!pip install -q "transformers>=4.44,<5" "peft>=0.13,<1" "datasets>=2.20,<4" "accelerate>=0.33"
!pip uninstall -y -q torchao


Importamos lo necesario y comprobamos las versiones que quedaron cargadas. `ast` lo usamos para
leer el nombre y la firma de la función, `hashlib` para buscar duplicados, `subprocess` para
ejecutar el código generado y `re` para extraer los bloques de código del texto que devuelve el
modelo. `pandas` y `scipy` son solo para las tablas y la comparación de distribuciones.


In [ ]:
import ast
import gc
import hashlib
import json
import os
import random
import re
import subprocess
import sys
import tempfile
import time
import warnings

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

import datasets
import peft
import transformers
from datasets import load_dataset
from google.colab import drive
from peft import LoraConfig, PeftModel, get_peft_model
from scipy.stats import ks_2samp
from transformers import AutoModelForCausalLM, AutoTokenizer, get_cosine_schedule_with_warmup

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("Entorno cargado")
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("peft:", peft.__version__)
print("numpy:", np.__version__)


Detectamos la GPU y elegimos la precisión mixta mirando el compute capability: bf16 solo en GPUs
Ampere o superiores (L4, A100), fp16 con `GradScaler` en T4 y en tarjetas más viejas. En la misma
celda dejamos las constantes del proyecto, incluidos los nombres y las revisiones fijas del
dataset y de los dos modelos.


In [ ]:
SEED = 42
MAX_LEN = 512
MAX_NEW_TOKENS = 512
BATCH_SIZE = 16

DATASET_ID = "google-research-datasets/mbpp"
DATASET_REVISION = "4bb6404fdc6cacfda99d4ac4205087b89d32030c"

QWEN_ID = "Qwen/Qwen2.5-1.5B"
QWEN_REVISION = "8faed761d45a263340a0528343f099c05c9a4323"
SMOL_ID = "HuggingFaceTB/SmolLM2-1.7B"
SMOL_REVISION = "effd688a12921b4cc83e3312b6feb579f70f9c71"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if DEVICE == "cuda":
    props = torch.cuda.get_device_properties(0)
    SOPORTA_BF16 = props.major >= 8
else:
    props = None
    SOPORTA_BF16 = False

if SOPORTA_BF16:
    AMP_DTYPE = torch.bfloat16
    NOMBRE_AMP = "bfloat16"
    USE_SCALER = False
else:
    AMP_DTYPE = torch.float16
    NOMBRE_AMP = "float16"
    USE_SCALER = True

if DEVICE == "cuda":
    print("GPU:", props.name)
    print("VRAM:", round(props.total_memory / 1024 ** 3, 1), "GB")
    print("Compute capability:", props.major, ".", props.minor)
else:
    print("No se detectó GPU: el entrenamiento en CPU va a ser muy lento")

print("Precisión mixta:", NOMBRE_AMP)
print("GradScaler:", USE_SCALER)
print("Máximo de tokens por ejemplo:", MAX_LEN)
print("Máximo de tokens nuevos al generar:", MAX_NEW_TOKENS)
print("Batch de evaluación:", BATCH_SIZE)


Montamos Drive y creamos las carpetas donde quedan los adaptadores y los resultados. Si la
sesión de Colab se cae, volvemos a ejecutar y lo encontramos todo ahí.


In [ ]:
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/microproyecto_3"
RESULTADOS_DIR = os.path.join(DRIVE_DIR, "resultados")
CHECKPOINT_QWEN = os.path.join(DRIVE_DIR, "checkpoint_qwen2.5_1.5b_lora")
CHECKPOINT_SMOL = os.path.join(DRIVE_DIR, "checkpoint_smollm2_1.7b_lora")

os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(RESULTADOS_DIR, exist_ok=True)

print("Carpeta de trabajo:", DRIVE_DIR)
print("Resultados:", RESULTADOS_DIR)
print("Adaptador de Qwen:", CHECKPOINT_QWEN)
print("Adaptador de SmolLM2:", CHECKPOINT_SMOL)

if "HF_TOKEN" in os.environ:
    print("HF_TOKEN: configurado")
else:
    print("HF_TOKEN: no está en el entorno, las descargas públicas funcionan igual")


Fijamos la semilla para que los resultados sean reproducibles y definimos la función que libera
la memoria de la GPU entre un modelo y el siguiente.


In [ ]:
def fijar_semilla(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


def liberar_vram(modelo=None, optimizador=None):
    if modelo is not None:
        del modelo
    if optimizador is not None:
        del optimizador
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.reset_peak_memory_stats()


fijar_semilla()
print("Semilla:", SEED)


## 3. Carga del dataset y exploración

Cargamos MBPP desde Hugging Face con la revisión fija de la sección anterior y lo miramos a fondo:
qué trae cada problema, cuánto miden los enunciados y las soluciones, cómo están escritas, qué
palabras se repiten y cómo son los `assert`. El análisis se hace sobre `train` y `validation`; el
split `test` solo se cuenta y se usa para comprobar que no haya problemas repetidos, porque queda
reservado para la evaluación final.


Cargamos el dataset y revisamos los splits y las columnas que trae.


In [ ]:
dataset = load_dataset(DATASET_ID, "full", revision=DATASET_REVISION)

print("Splits del dataset:")
for nombre in dataset:
    cuantos = len(dataset[nombre])
    if cuantos == 1:
        print(" ", nombre, "->", cuantos, "ejemplo")
    else:
        print(" ", nombre, "->", cuantos, "ejemplos")

print()
print("Columnas:", dataset["train"].column_names)


Comparamos los tamaños con lo que dice el enunciado: 500 problemas de `test` y el resto repartido
entre `train` y `validation`.


In [ ]:
TAMANOS_ESPERADOS = {"train": 374, "validation": 90, "test": 500}

for nombre in TAMANOS_ESPERADOS:
    esperado = TAMANOS_ESPERADOS[nombre]
    obtenido = len(dataset[nombre])
    if obtenido == esperado:
        print(nombre, "OK:", obtenido, "ejemplos")
    else:
        print(nombre, "distinto: se esperaban", esperado, "y hay", obtenido)

print()
print("Total de problemas para evaluar:", len(dataset["test"]))


Mostramos dos problemas de `train` completos: el enunciado, la solución de referencia, los casos
de prueba y el código de preparación que traen algunos ejercicios.


In [ ]:
for indice in [0, 1]:
    ejemplo = dataset["train"][indice]
    print("=" * 70)
    print("task_id:", ejemplo["task_id"])
    print()
    print("Enunciado:")
    print(ejemplo["text"])
    print()
    print("Solución de referencia:")
    print(ejemplo["code"])
    print()
    print("Casos de prueba:")
    for linea in ejemplo["test_list"]:
        print(" ", linea)
    print()
    print("Código de preparación:")
    if ejemplo["test_setup_code"].strip() == "":
        print("  (vacío)")
    else:
        print(ejemplo["test_setup_code"])
    print()


Revisamos que ningún `task_id` se repita entre particiones. Si `train` y `test` compartieran
problemas, el pass@1 saldría inflado porque el modelo habría entrenado con ellos.


In [ ]:
ids_train = []
for ejemplo in dataset["train"]:
    ids_train.append(ejemplo["task_id"])

ids_validation = []
for ejemplo in dataset["validation"]:
    ids_validation.append(ejemplo["task_id"])

ids_test = []
for ejemplo in dataset["test"]:
    ids_test.append(ejemplo["task_id"])

repetidos_train_validation = 0
for ident in ids_train:
    if ident in ids_validation:
        repetidos_train_validation = repetidos_train_validation + 1

repetidos_train_test = 0
for ident in ids_train:
    if ident in ids_test:
        repetidos_train_test = repetidos_train_test + 1

repetidos_validation_test = 0
for ident in ids_validation:
    if ident in ids_test:
        repetidos_validation_test = repetidos_validation_test + 1


def imprimir_split(nombre, ids):
    if len(ids) == 1:
        print(f"{nombre:<11}{len(ids):>4} problema, del {min(ids)} al {max(ids)}")
    else:
        print(f"{nombre:<11}{len(ids):>4} problemas, del {min(ids)} al {max(ids)}")


imprimir_split("train", ids_train)
imprimir_split("validation", ids_validation)
imprimir_split("test", ids_test)
print()
print("Repetidos entre train y validation:", repetidos_train_validation)
print("Repetidos entre train y test:", repetidos_train_test)
print("Repetidos entre validation y test:", repetidos_validation_test)


Sacamos la tabla descriptiva de `train` y de `validation` con cuartiles, desviación estándar y
extremos. Es la misma idea que un `describe()` de pandas: resumir cada variable en una sola vista
para comparar los dos splits de un vistazo.


In [ ]:
def promedio(valores):
    return sum(valores) / len(valores)


def percentil(valores, fraccion):
    ordenados = sorted(valores)
    posicion = int(len(ordenados) * fraccion)
    return ordenados[posicion]


largos_texto_train = []
largos_codigo_train = []
palabras_train = []
lineas_train = []
for ejemplo in dataset["train"]:
    largos_texto_train.append(len(ejemplo["text"]))
    largos_codigo_train.append(len(ejemplo["code"]))
    palabras_train.append(len(ejemplo["text"].split()))
    lineas_train.append(len(ejemplo["code"].splitlines()))

largos_texto_validation = []
largos_codigo_validation = []
palabras_validation = []
lineas_validation = []
for ejemplo in dataset["validation"]:
    largos_texto_validation.append(len(ejemplo["text"]))
    largos_codigo_validation.append(len(ejemplo["code"]))
    palabras_validation.append(len(ejemplo["text"].split()))
    lineas_validation.append(len(ejemplo["code"].splitlines()))

tabla_train = pd.DataFrame({
    "enunciado_car": largos_texto_train,
    "enunciado_pal": palabras_train,
    "solucion_car": largos_codigo_train,
    "solucion_lin": lineas_train,
})

tabla_validation = pd.DataFrame({
    "enunciado_car": largos_texto_validation,
    "enunciado_pal": palabras_validation,
    "solucion_car": largos_codigo_validation,
    "solucion_lin": lineas_validation,
})

print("train (n =", len(tabla_train), ")")
print(tabla_train.describe().round(1).to_string())
print()
print("validation (n =", len(tabla_validation), ")")
print(tabla_validation.describe().round(1).to_string())


Comparamos los dos splits con diagramas de caja. La caja muestra la mediana y el rango donde está
la mitad central de los datos, y los puntos sueltos son problemas atípicos.


In [ ]:
plt.figure(figsize=(10, 3))

plt.subplot(1, 2, 1)
plt.boxplot([largos_texto_train, largos_texto_validation])
plt.xticks([1, 2], ["train", "validation"])
plt.title("Enunciados")
plt.ylabel("Caracteres")

plt.subplot(1, 2, 2)
plt.boxplot([largos_codigo_train, largos_codigo_validation])
plt.xticks([1, 2], ["train", "validation"])
plt.title("Soluciones")
plt.ylabel("Caracteres")

plt.tight_layout()
plt.show()


Dibujamos la curva acumulada de las longitudes: para cada valor del eje x muestra qué porcentaje
de los problemas mide eso o menos. Sirve para decidir cuánto contexto necesitamos.


In [ ]:
def acumulada(valores):
    ordenados = sorted(valores)
    x = ordenados
    y = []
    for i in range(len(ordenados)):
        y.append((i + 1) / len(ordenados))
    return x, y


x_train, y_train = acumulada(largos_texto_train)
x_val, y_val = acumulada(largos_texto_validation)

plt.figure(figsize=(10, 3))

plt.subplot(1, 2, 1)
plt.plot(x_train, y_train, label="train")
plt.plot(x_val, y_val, linestyle="--", label="validation")
plt.axvline(percentil(largos_texto_train, 0.95), linestyle=":", color="gray")
plt.title("Enunciados")
plt.xlabel("Caracteres")
plt.ylabel("Proporción acumulada")
plt.legend()

x_train, y_train = acumulada(largos_codigo_train)
x_val, y_val = acumulada(largos_codigo_validation)

plt.subplot(1, 2, 2)
plt.plot(x_train, y_train, label="train")
plt.plot(x_val, y_val, linestyle="--", label="validation")
plt.axvline(percentil(largos_codigo_train, 0.95), linestyle=":", color="gray")
plt.title("Soluciones")
plt.xlabel("Caracteres")
plt.legend()

plt.tight_layout()
plt.show()

print("p95 del enunciado en train:", percentil(largos_texto_train, 0.95), "caracteres")
print("p95 de la solución en train:", percentil(largos_codigo_train, 0.95), "caracteres")


Miramos el vocabulario de los enunciados: cuántas palabras distintas usan, cuáles se repiten y
cuántos problemas empiezan con la misma fórmula. También revisamos si el nombre de la función
aparece en el texto, porque de eso depende el prompt.


In [ ]:
PALABRAS_VACIAS = [
    "a", "an", "the", "to", "of", "and", "in", "for", "is", "that", "with", "from",
    "as", "on", "by", "it", "this", "be", "are", "given", "write", "function", "python",
]

conteo_palabras = {}
todas_las_palabras = set()
for ejemplo in dataset["train"]:
    for palabra in re.findall(r"[a-z]+", ejemplo["text"].lower()):
        todas_las_palabras.add(palabra)
        if palabra in PALABRAS_VACIAS:
            continue
        if palabra in conteo_palabras:
            conteo_palabras[palabra] = conteo_palabras[palabra] + 1
        else:
            conteo_palabras[palabra] = 1

palabras_ordenadas = sorted(conteo_palabras, key=conteo_palabras.get, reverse=True)
top_palabras = palabras_ordenadas[:15]

empieza_to = 0
empieza_that = 0
empieza_python = 0
for ejemplo in dataset["train"]:
    texto = ejemplo["text"].lower().strip()
    if texto.startswith("write a function to"):
        empieza_to = empieza_to + 1
    if texto.startswith("write a function that"):
        empieza_that = empieza_that + 1
    if texto.startswith("write a python function"):
        empieza_python = empieza_python + 1

nombres_en_texto = 0
for ejemplo in dataset["train"]:
    nombre = ""
    for linea in ejemplo["test_list"]:
        arbol = ast.parse(linea)
        nodo = arbol.body[0]
        if isinstance(nodo, ast.Assert) and isinstance(nodo.test, ast.Compare):
            izquierda = nodo.test.left
            if isinstance(izquierda, ast.Call) and isinstance(izquierda.func, ast.Name):
                nombre = izquierda.func.id
                break
    if nombre != "" and nombre.lower() in ejemplo["text"].lower():
        nombres_en_texto = nombres_en_texto + 1

print("Palabras distintas en total:", len(todas_las_palabras))
print("Palabras distintas sin contar las vacías:", len(conteo_palabras))
print()
print("Palabras con contenido más frecuentes:")
for palabra in top_palabras:
    print(" ", palabra, "->", conteo_palabras[palabra])
print()
print("Enunciados que empiezan con 'write a function to':", empieza_to)
print("Enunciados que empiezan con 'write a function that':", empieza_that)
print("Enunciados que empiezan con 'write a python function':", empieza_python)
print("Enunciados que mencionan el nombre de la función:", nombres_en_texto, "de", len(dataset["train"]))


Dibujamos las palabras con contenido más frecuentes. Las palabras vacías y las de la plantilla
(“write”, “function”, “python”) quedaron fuera del conteo para que se vea el tema de cada
problema y no el formato de la instrucción.


In [ ]:
etiquetas = []
valores = []
for palabra in top_palabras:
    etiquetas.append(palabra)
    valores.append(conteo_palabras[palabra])

plt.figure(figsize=(7, 4))
plt.barh(etiquetas, valores)
plt.gca().invert_yaxis()
plt.xlabel("Apariciones en train")
plt.title("Palabras con contenido más frecuentes")
plt.tight_layout()
plt.show()


Medimos cuántas líneas tiene cada solución de `train`. Es otra forma de ver el tamaño del target:
una solución de una línea y otra de treinta no cuestan lo mismo de generar.


In [ ]:
print("Líneas por solución en train")
print("  mínimo:", min(lineas_train))
print("  promedio:", round(promedio(lineas_train), 1))
print("  mediana:", percentil(lineas_train, 0.50))
print("  p95:", percentil(lineas_train, 0.95))
print("  máximo:", max(lineas_train))

plt.figure(figsize=(7, 3))
plt.hist(lineas_train, bins=20)
plt.title("Líneas por solución en train")
plt.xlabel("Líneas")
plt.ylabel("Problemas")
plt.tight_layout()
plt.show()


Revisamos qué tan completas son las soluciones de referencia. Si varias traen imports, clases o
funciones auxiliares, el modelo tiene que aprender a generarlas y no basta con el cuerpo de una
sola función.


In [ ]:
def contar_estructura(particion):
    conteos = {"una función": 0, "varias funciones": 0, "con imports": 0, "con clases": 0, "sin leer": 0}
    for ejemplo in particion:
        try:
            arbol = ast.parse(ejemplo["code"])
        except SyntaxError:
            conteos["sin leer"] = conteos["sin leer"] + 1
            continue

        imports = 0
        clases = 0
        funciones = 0
        for nodo in arbol.body:
            if isinstance(nodo, ast.Import):
                imports = imports + 1
            if isinstance(nodo, ast.ImportFrom):
                imports = imports + 1
            if isinstance(nodo, ast.ClassDef):
                clases = clases + 1
            if isinstance(nodo, ast.FunctionDef):
                funciones = funciones + 1

        if funciones == 1:
            conteos["una función"] = conteos["una función"] + 1
        if funciones > 1:
            conteos["varias funciones"] = conteos["varias funciones"] + 1
        if imports > 0:
            conteos["con imports"] = conteos["con imports"] + 1
        if clases > 0:
            conteos["con clases"] = conteos["con clases"] + 1
    return conteos


conteos_train = contar_estructura(dataset["train"])
conteos_validation = contar_estructura(dataset["validation"])

print("train:", conteos_train)
print("validation:", conteos_validation)

categorias = ["una función", "varias funciones", "con imports", "con clases"]

porcentaje_train = []
porcentaje_validation = []
for categoria in categorias:
    porcentaje_train.append(100 * conteos_train[categoria] / len(dataset["train"]))
    porcentaje_validation.append(100 * conteos_validation[categoria] / len(dataset["validation"]))

plt.figure(figsize=(8, 3.5))
plt.barh(categorias, porcentaje_train, label="train")
plt.barh(categorias, porcentaje_validation, label="validation", alpha=0.6)
plt.xlabel("Porcentaje de problemas")
plt.title("Estructura del código de referencia")
plt.legend()
plt.tight_layout()
plt.show()


Contamos qué construcciones de Python aparecen en las soluciones: ciclos, condicionales, clases y
recursión. Es lo que el modelo tiene que aprender a escribir para resolver los problemas.


In [ ]:
con_for = 0
con_while = 0
con_if = 0
con_recursion = 0
con_clase = 0

for ejemplo in dataset["train"]:
    arbol = ast.parse(ejemplo["code"])

    nombres_funciones = []
    for nodo in arbol.body:
        if isinstance(nodo, ast.FunctionDef):
            nombres_funciones.append(nodo.name)

    usa_for = False
    usa_while = False
    usa_if = False
    usa_recursion = False
    usa_clase = False

    for nodo in ast.walk(arbol):
        if isinstance(nodo, ast.For):
            usa_for = True
        if isinstance(nodo, ast.While):
            usa_while = True
        if isinstance(nodo, ast.If):
            usa_if = True
        if isinstance(nodo, ast.ClassDef):
            usa_clase = True
        if isinstance(nodo, ast.Call) and isinstance(nodo.func, ast.Name):
            if nodo.func.id in nombres_funciones:
                usa_recursion = True

    if usa_for:
        con_for = con_for + 1
    if usa_while:
        con_while = con_while + 1
    if usa_if:
        con_if = con_if + 1
    if usa_recursion:
        con_recursion = con_recursion + 1
    if usa_clase:
        con_clase = con_clase + 1

print("Soluciones de train que usan:")
print("  for:", con_for)
print("  while:", con_while)
print("  if:", con_if)
print("  recursión:", con_recursion)
print("  clases:", con_clase)

etiquetas = ["for", "while", "if", "recursión", "clases"]
valores = [con_for, con_while, con_if, con_recursion, con_clase]

plt.figure(figsize=(8, 3))
plt.barh(etiquetas, valores)
plt.gca().invert_yaxis()
plt.xlabel("Problemas de train")
plt.title("Construcciones usadas en las soluciones")
plt.tight_layout()
plt.show()


Revisamos la calidad del texto: si hay soluciones con tabs en vez de espacios, si alguna imprime o
pide datos, si hay soluciones de una sola línea y si se repite algún enunciado o algún código. Al
final comparamos las distribuciones de `train` y `validation` con una prueba de
Kolmogorov-Smirnov.


In [ ]:
con_tabs = 0
con_print = 0
con_input = 0
una_linea = 0
for ejemplo in dataset["train"]:
    if "\t" in ejemplo["code"]:
        con_tabs = con_tabs + 1
    if "print(" in ejemplo["code"]:
        con_print = con_print + 1
    if "input(" in ejemplo["code"]:
        con_input = con_input + 1
    if len(ejemplo["code"].splitlines()) == 1:
        una_linea = una_linea + 1


def contar_duplicados(campo, normalizar):
    vistos = {}
    for nombre in ["train", "validation"]:
        for ejemplo in dataset[nombre]:
            valor = ejemplo[campo]
            if normalizar:
                valor = " ".join(valor.lower().split())
            clave = hashlib.md5(valor.encode()).hexdigest()
            if clave in vistos:
                vistos[clave] = vistos[clave] + 1
            else:
                vistos[clave] = 1
    repetidos = 0
    for clave in vistos:
        if vistos[clave] > 1:
            repetidos = repetidos + 1
    return repetidos


duplicados_texto_exactos = contar_duplicados("text", False)
duplicados_texto_normalizados = contar_duplicados("text", True)
duplicados_codigo_exactos = contar_duplicados("code", False)
duplicados_codigo_normalizados = contar_duplicados("code", True)

ks_texto = ks_2samp(largos_texto_train, largos_texto_validation)
ks_codigo = ks_2samp(largos_codigo_train, largos_codigo_validation)

print("Soluciones de train con tabs:", con_tabs, "de", len(dataset["train"]))
print("Soluciones de train con print(:", con_print)
print("Soluciones de train con input(:", con_input)
print("Soluciones de train de una sola línea:", una_linea)
print()
print("Duplicados entre train y validation")
print("  enunciados exactos:", duplicados_texto_exactos)
print("  enunciados normalizados:", duplicados_texto_normalizados)
print("  códigos exactos:", duplicados_codigo_exactos)
print("  códigos normalizados:", duplicados_codigo_normalizados)
print()
print("Kolmogorov-Smirnov entre train y validation")
print("  enunciado: estadístico", round(ks_texto.statistic, 3), "p-valor", round(ks_texto.pvalue, 3))
print("  solución: estadístico", round(ks_codigo.statistic, 3), "p-valor", round(ks_codigo.pvalue, 3))


Analizamos los `assert`: cuántos trae cada problema y qué tipo de valor esperan. Si comparan
listas o diccionarios, la verificación tiene que ejecutar el código de verdad y no basta con
comparar texto.


In [ ]:
total_asserts = 0
min_asserts = 100
max_asserts = 0
con_setup_total = 0
tipos_valor = {}

for nombre in ["train", "validation"]:
    for ejemplo in dataset[nombre]:
        cuantos = len(ejemplo["test_list"])
        if cuantos < min_asserts:
            min_asserts = cuantos
        if cuantos > max_asserts:
            max_asserts = cuantos
        if ejemplo["test_setup_code"].strip() != "":
            con_setup_total = con_setup_total + 1
        for linea in ejemplo["test_list"]:
            total_asserts = total_asserts + 1
            arbol = ast.parse(linea)
            nodo = arbol.body[0]
            if isinstance(nodo, ast.Assert):
                comparacion = nodo.test
                if isinstance(comparacion, ast.Compare):
                    valor = comparacion.comparators[0]
                else:
                    valor = comparacion
                if isinstance(valor, ast.Constant):
                    tipo = "constantes"
                elif isinstance(valor, ast.List):
                    tipo = "listas"
                elif isinstance(valor, ast.Tuple):
                    tipo = "tuplas"
                elif isinstance(valor, ast.Dict):
                    tipo = "diccionarios"
                elif isinstance(valor, ast.Set):
                    tipo = "conjuntos"
                elif isinstance(valor, ast.UnaryOp):
                    tipo = "valores negativos"
                else:
                    tipo = "otros"
                if tipo in tipos_valor:
                    tipos_valor[tipo] = tipos_valor[tipo] + 1
                else:
                    tipos_valor[tipo] = 1

tipos_ordenados = sorted(tipos_valor, key=tipos_valor.get, reverse=True)

print("Assert en train y validation:", total_asserts)
print("Assert por problema: mínimo", min_asserts, "máximo", max_asserts)
print("Problemas con test_setup_code:", con_setup_total)
print()
print("Tipo de valor esperado en assert:")
for tipo in tipos_ordenados:
    print(" ", tipo, "->", tipos_valor[tipo])


Buscamos los casos extremos de `train`: el enunciado más largo, la solución más larga y el único
problema que trae `test_setup_code`. Los extremos son los que pueden quedar truncados o romper la
ejecución de los `assert`.


In [ ]:
indice_texto = largos_texto_train.index(max(largos_texto_train))
indice_codigo = largos_codigo_train.index(max(largos_codigo_train))

print("Enunciado más largo de train")
print("  task_id:", dataset["train"][indice_texto]["task_id"])
print("  caracteres:", largos_texto_train[indice_texto])
print(" ", dataset["train"][indice_texto]["text"])
print()
print("Solución más larga de train")
print("  task_id:", dataset["train"][indice_codigo]["task_id"])
print("  caracteres:", largos_codigo_train[indice_codigo])
print("  líneas:", lineas_train[indice_codigo])

indice_setup = None
for i in range(len(dataset["train"])):
    if dataset["train"][i]["test_setup_code"].strip() != "":
        indice_setup = i
        break

print()
if indice_setup is None:
    print("Ningún problema de train trae test_setup_code")
else:
    print("Problema con test_setup_code")
    print("  task_id:", dataset["train"][indice_setup]["task_id"])
    print(dataset["train"][indice_setup]["test_setup_code"])


Revisamos la columna `challenge_test_list`, que trae pruebas adicionales, y miramos un ejemplo del
split `prompt`, que no usamos ni para entrenar ni para evaluar.


In [ ]:
con_challenge = 0
for ejemplo in dataset["train"]:
    if len(ejemplo["challenge_test_list"]) > 0:
        con_challenge = con_challenge + 1

print("Problemas de train con challenge_test_list:", con_challenge, "de", len(dataset["train"]))

ejemplo_prompt = dataset["prompt"][0]
print()
print("Un ejemplo del split prompt:")
print("task_id:", ejemplo_prompt["task_id"])
print()
print(ejemplo_prompt["text"])
print()
print(ejemplo_prompt["code"])


Mostramos un problema de `validation` para confirmar que tiene el mismo formato que `train`.


In [ ]:
ejemplo_validacion = dataset["validation"][0]
print("task_id:", ejemplo_validacion["task_id"])
print()
print(ejemplo_validacion["text"])
print()
print(ejemplo_validacion["code"])
print()
print("Primer caso de prueba:", ejemplo_validacion["test_list"][0])


### Lo que nos deja la exploración


**3.  `train` y `validation` ** El enunciado promedia 78.4 caracteres en `train` y
77.9 en `validation`, con p95 de 113 y 119. La solución promedia 178.4 y 182.4 caracteres, con
p95 de 412 y 447. Con 90 ejemplos la prueba tiene poca potencia, así que la lectura correcta es que no hay
evidencia suficiente para distinguir los splits, no que sean idénticos. Para lo que necesitamos
alcanza: si el modelo mejora en `validation`, es razonable esperar una mejora parecida en `test`.

**4. Los enunciados son plantillas.** De los 374 problemas, 247 empiezan literalmente con “write
a function to”, 8 con “write a function that” y 115 con “write a python function”; los 4
restantes son variantes menores, como un título en mayúsculas. El vocabulario es pequeño: 573
palabras distintas en total y 551 sin contar las vacías y las de la plantilla. Las palabras con
contenido más frecuentes son `find` (122), `list` (94), `number` (74), `string` (68), `check`
(54), `tuple` (50) y `sum` (41). Esa uniformidad tiene dos consecuencias. Primero, el modelo no
necesita interpretar instrucciones muy diversas: aprende un molde y lo repite, así que el pass@1
dependerá sobre todo de resolver el problema y no de entender el enunciado. Segundo, el nombre
de la función objetivo solo aparece en 13 de los 374 enunciados (3.5 %), aunque los `assert` la
llaman por su nombre; sin la firma en el prompt, el modelo tendría que adivinarlo y el pass@1
caería por una razón que no es de programación. Por eso el prompt incluye el nombre y la firma, y
solo eso.

**5. Las soluciones son cortas.** La mediana es de 145 caracteres y 5 líneas; el p95, de 412
caracteres y 16 líneas; la más larga llega a 908 caracteres y 29 líneas. Con estos tamaños, un
límite de 512 tokens para el target y de 512 tokens nuevos al generar deja margen de sobra: el
p95 queda muy por debajo del límite y solo la cola se le acerca. Si el límite bajara a 256, las
soluciones del extremo podrían quedar truncadas y el problema se contaría como incorrecto por el
recorte y no por el modelo. Ese es el argumento para fijar ese límite, y el conteo exacto en tokens
se confirma en la sección 4 con cada tokenizador.

**6. Qué tiene que generar el modelo.** De las 374 soluciones de `train`, 360 son una sola
función, 14 tienen más de una y 87 traen imports; 2 declaran clases. Además, 23 soluciones son
recursivas, 122 usan un ciclo `for`, 32 un `while` y 165 un `if`. Esto explica por qué el target
es el código de referencia completo y no solo el cuerpo de la función: si entrenáramos con el
cuerpo, el modelo nunca aprendería a escribir los imports, las clases ni las funciones auxiliares
que necesitan esos problemas.

**7. El estilo del código también importa.** 41 soluciones (11.0 %) usan tabs para indentar,
ninguna usa `print(` ni `input(`, y ninguna cabe en una sola línea. El modelo puede imitar los
tabs, así que el ejecutor tiene que aceptarlos y el tokenizador no puede asumir espacios. Que el
corpus no imprima ni pida datos es una buena noticia: el código generado no debería llenar la
salida ni quedarse esperando entrada, aunque nada impide que lo haga y por eso se ejecuta en un
subproceso aparte.

**8. Los `assert` son uniformes pero exigentes.** En `train` y `validation` hay 1392 `assert` y
todos los problemas traen exactamente 3. Los valores esperados se reparten así: 948 constantes,
294 listas, 93 tuplas, 36 diccionarios, 12 valores negativos, 6 conjuntos y 3 de otro tipo. Que
la mayoría sean constantes facilita la comparación, pero las listas, tuplas y diccionarios
obligan a ejecutar el código y comparar estructuras, no texto. Por eso la verificación de la
sección 6 corre el código generado en un subproceso y no intenta analizarlo estáticamente.
Además, el único problema con `test_setup_code` confirma que el código de preparación tiene que
ejecutarse antes de los `assert`, y en ese orden.

**9. Riesgos que deja la exploración.** El primero es el tamaño: la solución más larga (908
caracteres) es 6.3 veces la mediana de 145, así que cualquier límite de generación se define por
la cola y no por el caso típico. El segundo son los tabs y las clases: el código generado puede
ser sintácticamente válido y aun así fallar al ejecutarse por indentación o por nombres que no
existen. El tercero es el silencio de `validation`: solo 90 problemas, así que cada acierto mueve
el pass@1 1.1 puntos porcentuales y no alcanza para elegir hiperparámetros finos. Los tres se
cubren con el mismo diseño: límites holgados, ejecución aislada con timeout y decisiones tomadas
sobre el conjunto completo de validación, nunca sobre `test`.

**10. Limitaciones de esta exploración.** Las longitudes están en caracteres y palabras, no en
tokens: eso se mide en la sección 4 con cada tokenizador, porque depende del modelo. No medimos
dificultad ni calidad de las soluciones, solo su forma. `validation` tiene 90 ejemplos,
suficiente para comparar distribuciones gruesas pero no para detectar diferencias pequeñas. Y
`test` no se analizó a propósito: solo se contaron sus problemas y se revisaron sus
identificadores.

Con esto la sección 4 puede definir el prompt y el target con números en la mano: el formato de
instrucción tiene que aportar el nombre de la función, el target debe ser el código completo y el
límite de tokens puede fijarse en 512 sin truncar el 95 % de los casos.


## 4. Construcción de prompts y preprocesamiento

El enunciado deja libre el formato de instrucción y pide tres cosas: que el prompt describa el
problema, que el mismo formato se use en entrenamiento y evaluación, y que los tokens del prompt
queden enmascarados con `-100` para que la pérdida solo considere la solución. Aquí fijamos ese
formato, lo aplicamos a un ejemplo y medimos cuántos tokens ocupa con cada tokenizador.

El prompt lleva el enunciado y la firma de la función objetivo. Incluimos la firma porque los
`assert` llaman a la función por su nombre y el enunciado solo lo menciona en 13 de los 374
problemas; sin ella, el modelo tendría que adivinarlo. No incluimos el cuerpo de la solución, ni
los helpers, ni los `assert`, ni el `test_setup_code`: el prompt solo dice qué hay que resolver y
cómo se llama la función.

El formato queda así:

```text
### Problem
{enunciado}

### Signature
def nombre(argumentos):

### Solution
```

El target es el código de referencia completo (imports, clases, funciones auxiliares y función
objetivo) más el EOS. Lo dejamos completo porque 103 de las 374 soluciones de `train` no empiezan
con la función objetivo, y así el modelo aprende a escribir todo lo que la solución necesita.


Sacamos la firma leyendo el código con `ast`: el nombre sale de los `assert` y la línea del `def` se
corta en los dos puntos.


In [ ]:
FORMATO_PROBLEMA = "### Problem"
FORMATO_FIRMA = "### Signature"
FORMATO_SOLUCION = "### Solution"


def nombre_funcion_objetivo(ejemplo):
    arbol = ast.parse(ejemplo["code"])
    nombres = []
    for nodo in arbol.body:
        if isinstance(nodo, ast.FunctionDef):
            nombres.append(nodo.name)
    llamadas = []
    for linea in ejemplo["test_list"]:
        arbol_assert = ast.parse(linea)
        for nodo in ast.walk(arbol_assert):
            if isinstance(nodo, ast.Call) and isinstance(nodo.func, ast.Name):
                if nodo.func.id in nombres and nodo.func.id not in llamadas:
                    llamadas.append(nodo.func.id)
    if len(llamadas) > 0:
        return llamadas[0]
    return nombres[-1]


def extraer_firma(ejemplo):
    arbol = ast.parse(ejemplo["code"])
    objetivo = nombre_funcion_objetivo(ejemplo)
    for nodo in arbol.body:
        if isinstance(nodo, ast.FunctionDef) and nodo.name == objetivo:
            linea = ejemplo["code"].splitlines()[nodo.lineno - 1]
            corte = linea.index(":")
            return linea[: corte + 1].rstrip()
    return ""


for ident in [602, 606, 601]:
    for ejemplo in dataset["train"]:
        if ejemplo["task_id"] == ident:
            firma = extraer_firma(ejemplo)
            ast.parse(firma + "\n    pass")
            print("task_id:", ident)
            print("  función objetivo:", nombre_funcion_objetivo(ejemplo))
            print("  firma:", firma)
            print("  primera línea del código:", ejemplo["code"].splitlines()[0])
            print()


Armamos el prompt y el target de tres problemas: uno simple, uno que empieza con un import y uno
que empieza con una clase.


In [ ]:
def construir_prompt(ejemplo):
    firma = extraer_firma(ejemplo)
    prompt = FORMATO_PROBLEMA + "\n"
    prompt = prompt + ejemplo["text"].strip() + "\n\n"
    prompt = prompt + FORMATO_FIRMA + "\n"
    prompt = prompt + firma + "\n\n"
    prompt = prompt + FORMATO_SOLUCION + "\n"
    return prompt


def construir_solucion(ejemplo):
    return ejemplo["code"].rstrip() + "\n"


def buscar_ejemplo(particion, ident):
    for ejemplo in particion:
        if ejemplo["task_id"] == ident:
            return ejemplo
    return None


for ident in [602, 606, 601]:
    ejemplo = buscar_ejemplo(dataset["train"], ident)
    print("=" * 70)
    print(construir_prompt(ejemplo))
    print("--- target ---")
    print(construir_solucion(ejemplo))


Cargamos los dos tokenizadores con su revisión fija. Si no traen token de relleno les ponemos el
EOS, y los dejamos con padding a la izquierda porque así se genera en la evaluación. En
entrenamiento los lotes se arman a mano con relleno a la derecha.


In [ ]:
tokenizador_qwen = AutoTokenizer.from_pretrained(QWEN_ID, revision=QWEN_REVISION)
tokenizador_smol = AutoTokenizer.from_pretrained(SMOL_ID, revision=SMOL_REVISION)

TOKENIZADORES = [
    ("Qwen2.5-1.5B", tokenizador_qwen),
    ("SmolLM2-1.7B", tokenizador_smol),
]

for nombre, tokenizador in TOKENIZADORES:
    if tokenizador.pad_token is None:
        tokenizador.pad_token = tokenizador.eos_token
    tokenizador.padding_side = "left"

for nombre, tokenizador in TOKENIZADORES:
    print("=" * 50)
    print(nombre)
    print("  clase:", type(tokenizador).__name__)
    print("  vocabulario:", len(tokenizador))
    print("  BOS:", tokenizador.bos_token, "->", tokenizador.bos_token_id)
    print("  EOS:", tokenizador.eos_token, "->", tokenizador.eos_token_id)
    print("  PAD:", tokenizador.pad_token, "->", tokenizador.pad_token_id)
    print("  padding_side:", tokenizador.padding_side)


Tokenizamos un ejemplo completo y miramos dónde termina el prompt y dónde empieza la solución. El
EOS se agrega a mano porque ninguno de los dos tokenizadores lo pone, y queda supervisado.


In [ ]:
def tokenizar_ejemplo(ejemplo, tokenizador):
    prompt = construir_prompt(ejemplo)
    solucion = construir_solucion(ejemplo)
    ids_prompt = tokenizador(prompt, add_special_tokens=False)["input_ids"]
    ids_solucion = tokenizador(solucion, add_special_tokens=False)["input_ids"]
    ids_solucion = ids_solucion + [tokenizador.eos_token_id]
    ids = ids_prompt + ids_solucion
    labels = [-100] * len(ids_prompt) + ids_solucion
    return {
        "prompt": prompt,
        "solucion": solucion,
        "ids_prompt": ids_prompt,
        "ids_solucion": ids_solucion,
        "ids": ids,
        "labels": labels,
    }


ejemplo = buscar_ejemplo(dataset["train"], 602)
datos = tokenizar_ejemplo(ejemplo, tokenizador_qwen)

print("tokens del prompt:", len(datos["ids_prompt"]))
print("tokens de la solución (con EOS):", len(datos["ids_solucion"]))
print("tokens en total:", len(datos["ids"]))
print("labels en -100:", datos["labels"].count(-100))
print("labels supervisados:", len(datos["labels"]) - datos["labels"].count(-100))
print()
print("últimos tokens del prompt:", tokenizador_qwen.convert_ids_to_tokens(datos["ids_prompt"][-6:]))
print("primeros tokens de la solución:", tokenizador_qwen.convert_ids_to_tokens(datos["ids_solucion"][:6]))
print("último token de la solución:", tokenizador_qwen.convert_ids_to_tokens([datos["ids_solucion"][-1]]))
print()
print("final del prompt:", repr(datos["prompt"][-30:]))
print("inicio del target:", repr(datos["solucion"][:30]))
print()
enmascarados_prompt = datos["labels"][: len(datos["ids_prompt"])].count(-100)
print("el prompt está completo en -100:", enmascarados_prompt == len(datos["ids_prompt"]))
print("el EOS está supervisado:", datos["labels"][-1] == tokenizador_qwen.eos_token_id)
print("cabe en MAX_LEN:", len(datos["ids"]) <= MAX_LEN)


Comprobamos en `train` y `validation` que el prompt tokenizado sea exactamente el prefijo del texto
completo. Si no lo fuera, la máscara por longitud no coincidiría con la frontera real y el modelo
recibiría pérdida por tokens del enunciado. También revisamos que todas las firmas sean Python
válido y en cuántos problemas la función objetivo no es el último `def`.


In [ ]:
print("El prompt tokenizado es prefijo del texto completo")
for nombre, tokenizador in TOKENIZADORES:
    revisados = 0
    distintos = 0
    for particion in ["train", "validation"]:
        for ejemplo in dataset[particion]:
            datos = tokenizar_ejemplo(ejemplo, tokenizador)
            texto_completo = datos["prompt"] + datos["solucion"]
            ids_completos = tokenizador(texto_completo, add_special_tokens=False)["input_ids"]
            if ids_completos[: len(datos["ids_prompt"])] != datos["ids_prompt"]:
                distintos = distintos + 1
            revisados = revisados + 1
    print(" ", nombre, "->", distintos, "diferencias en", revisados, "ejemplos")

firmas_invalidas = 0
for particion in ["train", "validation"]:
    for ejemplo in dataset[particion]:
        firma = extraer_firma(ejemplo)
        try:
            ast.parse(firma + "\n    pass")
        except SyntaxError:
            firmas_invalidas = firmas_invalidas + 1

print("Firmas que no son Python válido:", firmas_invalidas)

no_ultimo = 0
ids_no_ultimo = []
for particion in ["train", "validation"]:
    for ejemplo in dataset[particion]:
        arbol = ast.parse(ejemplo["code"])
        defs = []
        for nodo in arbol.body:
            if isinstance(nodo, ast.FunctionDef):
                defs.append(nodo.name)
        if nombre_funcion_objetivo(ejemplo) != defs[-1]:
            no_ultimo = no_ultimo + 1
            ids_no_ultimo.append(ejemplo["task_id"])

print("Problemas donde la función objetivo no es el último def:", no_ultimo)
print("  task_id:", ids_no_ultimo)


Medimos cuántos tokens ocupa el prompt, la solución y el total en `train` y `validation` con cada
tokenizador.


In [ ]:
filas = []
superan_total = 0
for nombre_split in ["train", "validation"]:
    for nombre_modelo, tokenizador in TOKENIZADORES:
        largos_prompt = []
        largos_solucion = []
        largos_total = []
        superan = 0
        for ejemplo in dataset[nombre_split]:
            datos = tokenizar_ejemplo(ejemplo, tokenizador)
            largos_prompt.append(len(datos["ids_prompt"]))
            largos_solucion.append(len(datos["ids_solucion"]))
            largos_total.append(len(datos["ids"]))
            if len(datos["ids"]) > MAX_LEN:
                superan = superan + 1
        superan_total = superan_total + superan
        resumen_prompt = str(round(promedio(largos_prompt), 1))
        resumen_prompt = resumen_prompt + " / " + str(percentil(largos_prompt, 0.50))
        resumen_prompt = resumen_prompt + " / " + str(percentil(largos_prompt, 0.95))
        resumen_prompt = resumen_prompt + " / " + str(max(largos_prompt))
        resumen_solucion = str(round(promedio(largos_solucion), 1))
        resumen_solucion = resumen_solucion + " / " + str(percentil(largos_solucion, 0.50))
        resumen_solucion = resumen_solucion + " / " + str(percentil(largos_solucion, 0.95))
        resumen_solucion = resumen_solucion + " / " + str(max(largos_solucion))
        resumen_total = str(round(promedio(largos_total), 1))
        resumen_total = resumen_total + " / " + str(percentil(largos_total, 0.50))
        resumen_total = resumen_total + " / " + str(percentil(largos_total, 0.95))
        resumen_total = resumen_total + " / " + str(max(largos_total))
        filas.append({
            "split": nombre_split,
            "modelo": nombre_modelo,
            "prompt": resumen_prompt,
            "solución": resumen_solucion,
            "total": resumen_total,
            "superan 512": superan,
        })

tabla_tokens = pd.DataFrame(filas)
print("Cada medida es media / mediana / p95 / máximo, en tokens")
print(tabla_tokens.to_string(index=False))
print()
print("Ejemplos que superan MAX_LEN:", superan_total)


Dibujamos la curva acumulada del total de tokens. En el eje x está el tamaño del ejemplo y en el eje
y la proporción de problemas que caben en ese tamaño o menos. La línea roja marca el límite de 512.


In [ ]:
maximos = {}
superan_total = 0

plt.figure(figsize=(10, 3.5))
for posicion in range(len(TOKENIZADORES)):
    nombre_modelo, tokenizador = TOKENIZADORES[posicion]
    plt.subplot(1, 2, posicion + 1)
    for nombre_split in ["train", "validation"]:
        largos = []
        for ejemplo in dataset[nombre_split]:
            datos = tokenizar_ejemplo(ejemplo, tokenizador)
            largos.append(len(datos["ids"]))
            if len(datos["ids"]) > MAX_LEN:
                superan_total = superan_total + 1
        maximos[nombre_modelo + " " + nombre_split] = max(largos)
        x, y = acumulada(largos)
        plt.plot(x, y, label=nombre_split)
    plt.axvline(MAX_LEN, color="tab:red", linestyle="--", label="límite de " + str(MAX_LEN))
    plt.title(nombre_modelo)
    plt.xlabel("Tokens del prompt y la solución")
    plt.ylabel("Proporción acumulada")
    plt.legend()

plt.tight_layout()
plt.show()

print("Máximo de tokens por split y modelo:")
for clave in maximos:
    print(" ", clave, "->", maximos[clave])
print("Ejemplos que superan MAX_LEN:", superan_total)


Armamos un lote de cuatro ejemplos con relleno dinámico: el lote se estira hasta el ejemplo más
largo y los `labels` del relleno también quedan en `-100`. Así la pérdida no considera ni el prompt
ni el relleno.


In [ ]:
def preparar_lote(ejemplos, tokenizador):
    datos = []
    for ejemplo in ejemplos:
        datos.append(tokenizar_ejemplo(ejemplo, tokenizador))

    largo_maximo = 0
    for fila in datos:
        if len(fila["ids"]) > largo_maximo:
            largo_maximo = len(fila["ids"])

    if largo_maximo > MAX_LEN:
        raise ValueError("Un ejemplo del lote supera MAX_LEN: " + str(largo_maximo))

    input_ids = []
    attention_mask = []
    labels = []
    for fila in datos:
        relleno = largo_maximo - len(fila["ids"])
        input_ids.append(fila["ids"] + [tokenizador.pad_token_id] * relleno)
        attention_mask.append([1] * len(fila["ids"]) + [0] * relleno)
        labels.append(fila["labels"] + [-100] * relleno)

    lote = {
        "input_ids": torch.tensor(input_ids),
        "attention_mask": torch.tensor(attention_mask),
        "labels": torch.tensor(labels),
    }
    return lote


ejemplos_demo = []
for indice in range(4):
    ejemplos_demo.append(dataset["train"][indice])

lote = preparar_lote(ejemplos_demo, tokenizador_qwen)

print("input_ids:", tuple(lote["input_ids"].shape))
print("attention_mask:", tuple(lote["attention_mask"].shape))
print("labels:", tuple(lote["labels"].shape))
print()

filas = []
for i in range(len(ejemplos_demo)):
    etiquetas = lote["labels"][i].tolist()
    mascara = lote["attention_mask"][i].tolist()
    enmascarados = 0
    supervisados = 0
    relleno = 0
    for j in range(len(etiquetas)):
        if mascara[j] == 0:
            relleno = relleno + 1
        elif etiquetas[j] == -100:
            enmascarados = enmascarados + 1
        else:
            supervisados = supervisados + 1
    filas.append({
        "task_id": ejemplos_demo[i]["task_id"],
        "prompt en -100": enmascarados,
        "supervisados": supervisados,
        "relleno": relleno,
        "total": len(etiquetas),
    })

print(pd.DataFrame(filas).to_string(index=False))
print()
print("labels del primer ejemplo:")
print(lote["labels"][0].tolist())


Buscamos fugas: que la solución completa, los `assert` o el `test_setup_code` no aparezcan dentro
del prompt. También revisamos línea por línea el código de referencia, sin contar la firma, que es
la única parte del código que el prompt sí debe llevar.


In [ ]:
fugas_solucion = 0
fugas_assert = 0
fugas_setup = 0
lineas_repetidas = []

for particion in ["train", "validation"]:
    for ejemplo in dataset[particion]:
        prompt = construir_prompt(ejemplo)
        firma = extraer_firma(ejemplo)
        solucion = construir_solucion(ejemplo)

        if solucion.strip() in prompt:
            fugas_solucion = fugas_solucion + 1
        for linea in ejemplo["test_list"]:
            if linea.strip() in prompt:
                fugas_assert = fugas_assert + 1
        if ejemplo["test_setup_code"].strip() != "":
            if ejemplo["test_setup_code"].strip() in prompt:
                fugas_setup = fugas_setup + 1
        for linea in ejemplo["code"].splitlines():
            if linea.strip() == "":
                continue
            if linea.strip() == firma:
                continue
            if linea.strip() in prompt:
                lineas_repetidas.append((ejemplo["task_id"], linea.strip()))

print("Soluciones completas dentro del prompt:", fugas_solucion)
print("Assert dentro del prompt:", fugas_assert)
print("test_setup_code dentro del prompt:", fugas_setup)
print("Líneas de código (sin la firma) dentro del prompt:", len(lineas_repetidas))
for ident, linea in lineas_repetidas:
    print("  task_id", ident, "->", repr(linea))


### Lo que nos deja el preprocesamiento

**11. El prompt es una parte pequeña del ejemplo.** Con este formato, el prompt promedia 32.7
tokens con Qwen y 40.6 con SmolLM2 en `train`, y 32.4 y 40.2 en `validation`. El máximo es de 68 y
74 tokens. Frente a una solución que promedia 58.2 y 66.9 tokens, el prompt se queda con algo más
de un tercio del total: 36.0 % con Qwen y 37.8 % con SmolLM2. Eso es lo que queremos, porque el
prompt se enmascara y solo la solución aporta a la pérdida.

**12. La máscara cae justo en la frontera.** En el problema 602 el prompt ocupa 32 tokens y la
solución 43 (42 de código más el EOS). Los 32 primeros `labels` quedan en `-100` y los 43
restantes conservan su id, incluido el EOS. La comprobación sobre los 464 ejemplos da 0
diferencias: el prompt tokenizado es el prefijo exacto del texto completo con los dos
tokenizadores, así que la máscara por longitud no parte ningún token ni deja texto sin cubrir.

**13. 512 tokens sobran.** El total más largo es de 346 tokens (SmolLM2, `train`) y el de Qwen
llega a 321; en `validation` los máximos son 281 y 236. Ninguno de los 464 ejemplos supera el
límite y el margen es de 166 tokens, un 32 % de `MAX_LEN`. La mediana del total es de 78 tokens
con Qwen y 92 con SmolLM2, así que el límite lo fija la cola y no el caso típico. Con
`MAX_NEW_TOKENS = 512` pasa lo mismo: la solución más larga (308 tokens con SmolLM2) cabe entera.

**14. La firma no siempre es el último `def`.** Las 464 firmas son Python válido y el nombre de la
función objetivo siempre coincide con una función definida en el código. En 2 problemas (717 y
940) la función que llaman los `assert` aparece antes de una función auxiliar, así que quedarse
con el último `def` habría dado una firma equivocada. Por eso el nombre se saca de los `assert` y
no de la posición.

**15. No hay fuga en el prompt.** Ninguna solución completa, ningún `assert` y ningún
`test_setup_code` aparecen dentro del prompt de los 464 ejemplos. La única línea de código que
coincide es `return 20` en el problema 675, y coincide porque el enunciado dice literalmente “it
will return 20”: es una casualidad del texto, no una filtración de la solución.

**16. El relleno se paga por lote, no por corpus.** En el lote de cuatro ejemplos el más largo mide
183 tokens y el más corto 41, así que este último lleva 142 de relleno. Con relleno dinámico esa
diferencia se paga solo dentro del lote y no se reserva `MAX_LEN` para todos: en entrenamiento cada
lote se estira hasta su ejemplo más largo, y el relleno queda con máscara 0 y `-100` en los
`labels`. En generación se usa padding a la izquierda, que es lo que permite que los 16 prompts del
lote terminen en la misma columna.

Con esto el pipeline queda definido: `construir_prompt` y `construir_solucion` arman los textos,
`tokenizar_ejemplo` aplica la máscara y `preparar_lote` construye los tensores. La sección 5 lo
aplica a las tres particiones y deja escrito el rol de cada una.


## 5. Particiones (train / validation / test)

MBPP ya viene partido en tres conjuntos y no los tocamos: usamos los splits oficiales tal como
están. La única decisión nuestra es qué rol cumple cada uno. `train` se usa para el finetuning,
`validation` para monitorear el entrenamiento y elegir la mejor época, y `test` queda reservado
para la evaluación final: no se tokeniza ni se genera nada de él antes de la sección 7.

De `validation` apartamos 50 problemas fijos para revisar el avance después de cada época. Los
elegimos al azar con la semilla del proyecto para no quedarnos con un extremo del conjunto.


Dejamos escrito el rol de cada split y qué se permite hacer con él.


In [ ]:
filas_roles = [
    {
        "split": "train",
        "problemas": len(dataset["train"]),
        "rol": "entrenamiento",
        "uso": "finetuning con LoRA; nunca se evalúa",
    },
    {
        "split": "validation",
        "problemas": len(dataset["validation"]),
        "rol": "monitoreo",
        "uso": "pérdida y pass@1 para elegir época e hiperparámetros",
    },
    {
        "split": "test",
        "problemas": len(dataset["test"]),
        "rol": "evaluación final",
        "uso": "pass@1 final; no se usa para decidir nada",
    },
]

tabla_roles = pd.DataFrame(filas_roles)
print(tabla_roles.to_string(index=False))


Guardamos las listas de entrenamiento y validación y calculamos cuántos lotes tiene una época.


In [ ]:
EJEMPLOS_ENTRENAMIENTO = dataset["train"]
EJEMPLOS_VALIDACION = dataset["validation"]

lotes_por_epoca = len(EJEMPLOS_ENTRENAMIENTO) // BATCH_SIZE
sobrantes = len(EJEMPLOS_ENTRENAMIENTO) % BATCH_SIZE
if sobrantes > 0:
    lotes_por_epoca = lotes_por_epoca + 1

print("Ejemplos de entrenamiento:", len(EJEMPLOS_ENTRENAMIENTO))
print("Ejemplos de validación:", len(EJEMPLOS_VALIDACION))
print("Batch de entrenamiento:", BATCH_SIZE)
print("Lotes por época:", lotes_por_epoca)
if sobrantes == 0:
    print("Todos los lotes van completos")
else:
    print("Ejemplos en el último lote:", sobrantes)


Elegimos los 50 problemas de monitoreo con la semilla del proyecto y comprobamos que sean
distintos y que ninguno esté en entrenamiento.


In [ ]:
CANTIDAD_MONITOREO = 50

generador = random.Random(SEED)
indices_monitoreo = generador.sample(range(len(EJEMPLOS_VALIDACION)), CANTIDAD_MONITOREO)
indices_monitoreo = sorted(indices_monitoreo)

EJEMPLOS_MONITOREO = []
for indice in indices_monitoreo:
    EJEMPLOS_MONITOREO.append(EJEMPLOS_VALIDACION[indice])

ids_entrenamiento = []
for ejemplo in EJEMPLOS_ENTRENAMIENTO:
    ids_entrenamiento.append(ejemplo["task_id"])

ids_monitoreo = []
for ejemplo in EJEMPLOS_MONITOREO:
    ids_monitoreo.append(ejemplo["task_id"])

ids_en_entrenamiento = 0
for ident in ids_monitoreo:
    if ident in ids_entrenamiento:
        ids_en_entrenamiento = ids_en_entrenamiento + 1

print("Semilla:", SEED)
print("Problemas de validación para monitoreo:", len(EJEMPLOS_MONITOREO))
print("task_id distintos:", len(set(ids_monitoreo)))
print("task_id que también están en entrenamiento:", ids_en_entrenamiento)
print("task_id:", ids_monitoreo)


Calculamos cuántos tokens ve el modelo en una época para dimensionar el entrenamiento. El costo de
generar es aparte: aquí solo contamos lo que se recorre hacia adelante y hacia atrás.


In [ ]:
filas_presupuesto = []
for nombre, tokenizador in TOKENIZADORES:
    tokens_por_epoca = 0
    for ejemplo in EJEMPLOS_ENTRENAMIENTO:
        datos = tokenizar_ejemplo(ejemplo, tokenizador)
        tokens_por_epoca = tokens_por_epoca + len(datos["ids"])
    filas_presupuesto.append({
        "modelo": nombre,
        "tokens por época": tokens_por_epoca,
        "tokens en 3 épocas": tokens_por_epoca * 3,
        "tokens en 5 épocas": tokens_por_epoca * 5,
        "pasos en 5 épocas": lotes_por_epoca * 5,
    })

tabla_presupuesto = pd.DataFrame(filas_presupuesto)
print(tabla_presupuesto.to_string(index=False))


Comparamos los 50 de monitoreo con los 90 de validación: longitudes en tokens con Qwen y estructura
del código de referencia. Si se parecen, el pass@1 de los 50 sirve para decidir la época.


In [ ]:
largos_monitoreo = []
for ejemplo in EJEMPLOS_MONITOREO:
    datos = tokenizar_ejemplo(ejemplo, tokenizador_qwen)
    largos_monitoreo.append(len(datos["ids"]))

largos_validacion = []
for ejemplo in EJEMPLOS_VALIDACION:
    datos = tokenizar_ejemplo(ejemplo, tokenizador_qwen)
    largos_validacion.append(len(datos["ids"]))

print("Monitoreo (n = 50)")
print("  media:", round(promedio(largos_monitoreo), 1))
print("  mediana:", percentil(largos_monitoreo, 0.50))
print("  máximo:", max(largos_monitoreo))
print()
print("Validación (n = 90)")
print("  media:", round(promedio(largos_validacion), 1))
print("  mediana:", percentil(largos_validacion, 0.50))
print("  máximo:", max(largos_validacion))
print()
print("Estructura del código de referencia")
print("  monitoreo:", contar_estructura(EJEMPLOS_MONITOREO))
print("  validación:", contar_estructura(EJEMPLOS_VALIDACION))


### Lo que nos deja la partición

**17. Tres conjuntos con roles separados.** El dataset trae 374 problemas de entrenamiento, 90 de
validación y 500 de prueba. `train` solo se usa para el finetuning; `validation` para la pérdida y
el pass@1 de monitoreo; `test` queda intacto hasta la sección 7. La tabla de roles deja esa
separación escrita en el notebook y no solo en la cabeza de quien lo corre.

**18. Una época son 24 pasos.** Con batch de 16, los 374 ejemplos dan 23 lotes completos y uno de
6. Cinco épocas son 120 pasos de optimización, así que el bucle de entrenamiento es corto y se
puede seguir época por época sin problema.

**19. El entrenamiento no es el costo.** Una época recorre 33,984 tokens con Qwen y 40,201 con
SmolLM2; cinco épocas llegan a 169,920 y 201,005. Son cifras pequeñas para una GPU de 24 GB. El
costo real está en generar 500 soluciones por cada una de las cuatro configuraciones, que es lo que
justifica la generación por lotes y el recorte del prompt.

**20. Los 50 de monitoreo representan al conjunto.** Salieron 50 `task_id` distintos entre 511 y
600, ninguno de entrenamiento. En tokens son un poco más largos que los 90 (mediana 85 contra 77,
media 100.8 contra 92.7), pero el máximo es el mismo (236). La estructura también se parece: 4 de
50 tienen más de una función (8 %) contra 6 de 90 (6.7 %), y 7 de 50 traen imports (14 %) contra
13 de 90 (14.4 %). Con esa muestra el pass@1 de monitoreo sirve para comparar épocas, no para
sacar conclusiones finas.

**21. Qué se decide con esto.** La época y los hiperparámetros se eligen con la pérdida de los 90 y
el pass@1 de los 50; `test` no participa. Como cada acierto en 50 mueve el pass@1 dos puntos
porcentuales, solo miramos diferencias de varios problemas y desempatamos por la pérdida, como
estaba previsto. La lista de `test` se arma en la sección 7, cuando ya no hay nada que elegir.


## 6. Implementación de la evaluación pass@1

pass@1 es la proporción de problemas en los que el modelo acierta a la primera. Para cada problema
generamos una única solución con decodificación greedy, la ejecutamos y corremos sus `assert`; si
todos pasan, el problema cuenta como acierto. Al final dividimos los aciertos entre los problemas
evaluados.

La evaluación tiene cuatro pasos: generar las soluciones en lotes, extraer el código del texto que
devuelve el modelo, ejecutarlo en un entorno aislado y contar los aciertos. Aquí implementamos los
cuatro con la misma configuración para las cuatro configuraciones del proyecto: mismo prompt,
`max_new_tokens=512`, una sola solución y decodificación greedy. En esta sección no se descarga
ningún modelo ni se toca `test`: el sandbox se prueba con soluciones escritas a mano y la
generación se estrena en la sección 7.


Generamos varias soluciones a la vez para aprovechar la GPU. Los prompts de un lote tienen
longitudes distintas, así que el relleno va a la izquierda: de esa forma el último token real de
cada fila queda al final y el modelo puede continuar desde ahí. La `attention_mask` marca el
relleno y el recorte descarta el prompt para quedarnos solo con lo generado.


In [ ]:
@torch.no_grad()
def generar_soluciones(
    modelo,
    tokenizador,
    prompts,
    tamano_lote=BATCH_SIZE,
    max_new_tokens=MAX_NEW_TOKENS,
):
    tokenizador.padding_side = "left"
    if tokenizador.pad_token is None:
        tokenizador.pad_token = tokenizador.eos_token
    modelo.config.pad_token_id = tokenizador.pad_token_id
    modelo.eval()

    soluciones = []
    for inicio in tqdm(range(0, len(prompts), tamano_lote), desc="Generando", leave=False):
        bloque = prompts[inicio: inicio + tamano_lote]
        entradas = tokenizador(
            bloque,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LEN,
        )
        entradas = entradas.to(modelo.device)
        salida = modelo.generate(
            **entradas,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizador.pad_token_id,
        )
        generado = salida[:, entradas["input_ids"].shape[1]:]
        textos = tokenizador.batch_decode(generado, skip_special_tokens=True)
        for texto in textos:
            soluciones.append(texto)
    return soluciones


Antes de generar con un modelo, comprobamos que el tokenizador deja el relleno a la izquierda y
que la máscara lo marca. Usamos tres problemas de validación para no tocar `test`.


In [ ]:
prompts_demo = []
for ejemplo in EJEMPLOS_MONITOREO[:3]:
    prompts_demo.append(construir_prompt(ejemplo))

tokenizador_demo = tokenizador_qwen
tokenizador_demo.padding_side = "left"
entradas_demo = tokenizador_demo(
    prompts_demo,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=MAX_LEN,
)

print("input_ids:", tuple(entradas_demo["input_ids"].shape))
print("attention_mask:", tuple(entradas_demo["attention_mask"].shape))
print()

for i in range(len(prompts_demo)):
    ids = entradas_demo["input_ids"][i].tolist()
    mascara = entradas_demo["attention_mask"][i].tolist()
    reales = sum(mascara)
    relleno = len(ids) - reales
    print("prompt", i + 1, "->", reales, "tokens reales,", relleno, "de relleno")
    print("  máscara al inicio:", mascara[:4])
    print("  máscara al final:", mascara[-4:])
    print("  últimos tokens:", tokenizador_demo.convert_ids_to_tokens(ids[-4:]))

sin_relleno = tokenizador_demo(prompts_demo[0], add_special_tokens=False)["input_ids"]
ultimo_sin_relleno = tokenizador_demo.convert_ids_to_tokens(sin_relleno[-1:])
ultimo_en_lote = tokenizador_demo.convert_ids_to_tokens(entradas_demo["input_ids"][0].tolist()[-1:])
print()
print("último token sin relleno:", ultimo_sin_relleno)
print("último token en el lote:", ultimo_en_lote)


El modelo devuelve texto, no código. Puede encerrar la solución en un bloque markdown, repetir el
encabezado `### Solution`, escribir una frase antes del `def` o agregar una explicación al final.
La extracción se queda con el primer fragmento que es Python válido, corta en encabezados
posteriores como `### Explanation` o `### Test` y descarta lo que no es código, pero no corrige la
solución: si el modelo no escribe Python válido, el problema falla.


In [ ]:
def extraer_codigo(texto):
    texto = texto.replace("\r\n", "\n")

    partes = texto.split("```")
    if len(partes) >= 3:
        for posicion in range(1, len(partes), 2):
            bloque = partes[posicion]
            lineas = bloque.split("\n")
            if len(lineas) > 0:
                primera = lineas[0].strip().lower()
                if primera in ["python", "py", "python3"]:
                    lineas = lineas[1:]
            bloque = "\n".join(lineas).strip("\n")
            if bloque.strip() != "":
                texto = bloque
                break

    if "### Solution" in texto:
        texto = texto.split("### Solution")[-1]

    marcas = ["\n### Problem", "\n### Signature", "\n### Explanation", "\n### Test"]
    marcas.append("\n### Output")
    marcas.append("\n### Example")
    marcas.append("\n### Examples")
    marcas.append("\n### Note")
    marcas.append("\n### Code")
    marcas.append("\n### Answer")
    corte = len(texto)
    for marca in marcas:
        if marca in texto:
            posicion = texto.index(marca)
            if posicion > 0 and posicion < corte:
                corte = posicion
    texto = texto[:corte]

    lineas = texto.split("\n")
    for i in range(len(lineas)):
        if lineas[i].strip() == "":
            continue
        if lineas[i][0].isspace():
            continue
        candidato = "\n".join(lineas[i:]).strip("\n")
        try:
            ast.parse(candidato)
            return candidato + "\n"
        except SyntaxError:
            continue

    codigo = texto.strip("\n")
    return codigo + "\n"


textos = [
    "def doblar(x):\n    return 2 * x\n",
    "```python\ndef doblar(x):\n    return 2 * x\n```",
    "Aquí está la solución:\n\ndef doblar(x):\n    return 2 * x\n",
    "### Solution\ndef doblar(x):\n    return 2 * x\n\n### Problem\nOtro problema",
    "MAX = 10\n\ndef doblar(x):\n    return MAX * x\n",
]

for texto in textos:
    codigo = extraer_codigo(texto)
    print("---")
    print(repr(codigo))


Ejecutamos cada solución en un subproceso aparte, nunca en el kernel. El subproceso arranca con
`-I` (sin el directorio actual ni paquetes del usuario), corre en una carpeta temporal, usa un
`HOME` temporal y no recibe las variables con credenciales. El tiempo límite es de 5 segundos: un
bucle infinito se corta y el notebook sigue. El orden es código generado, `test_setup_code` y
`assert`, porque hay problemas cuyo setup usa clases que define la solución.


In [ ]:
def ejecutar_solucion(codigo, ejemplo, limite_segundos=5):
    if codigo.strip() == "":
        return {"pasa": False, "motivo": "vacio", "detalle": "sin código", "segundos": 0.0}

    preparacion = ejemplo["test_setup_code"].replace("\r\n", "\n").strip()

    partes = [codigo.rstrip()]
    if preparacion != "":
        partes.append(preparacion)
    for linea in ejemplo["test_list"]:
        partes.append(linea.strip())
    partes.append('print("__FIN__")')
    script = "\n\n".join(partes) + "\n"

    entorno = os.environ.copy()
    claves = ["HF_TOKEN", "HUGGING_FACE_HUB_TOKEN", "OPENAI_API_KEY", "GITHUB_TOKEN", "GH_TOKEN"]
    for clave in claves:
        if clave in entorno:
            del entorno[clave]

    inicio = time.time()
    with tempfile.TemporaryDirectory() as directorio:
        entorno["HOME"] = directorio
        try:
            proceso = subprocess.run(
                [sys.executable, "-I", "-c", script],
                capture_output=True,
                text=True,
                encoding="utf-8",
                errors="replace",
                timeout=limite_segundos,
                cwd=directorio,
                env=entorno,
            )
        except subprocess.TimeoutExpired:
            segundos = time.time() - inicio
            detalle = "más de " + str(limite_segundos) + " segundos"
            return {"pasa": False, "motivo": "timeout", "detalle": detalle, "segundos": segundos}
        except Exception as error:
            segundos = time.time() - inicio
            return {"pasa": False, "motivo": "entorno", "detalle": str(error), "segundos": segundos}

    segundos = time.time() - inicio
    detalle = proceso.stderr
    if detalle is None:
        detalle = ""
    if len(detalle) > 500:
        detalle = detalle[-500:]

    if proceso.returncode == 0 and "__FIN__" in proceso.stdout:
        return {"pasa": True, "motivo": "ok", "detalle": "", "segundos": segundos}
    if "AssertionError" in detalle:
        return {"pasa": False, "motivo": "assert", "detalle": detalle, "segundos": segundos}
    if "SyntaxError" in detalle or "IndentationError" in detalle:
        return {"pasa": False, "motivo": "sintaxis", "detalle": detalle, "segundos": segundos}
    return {"pasa": False, "motivo": "error", "detalle": detalle, "segundos": segundos}


Recorremos los problemas, generamos una solución por cada uno y ejecutamos sus `assert`. Cada
problema queda registrado con su `task_id`, si pasó, el motivo, el tiempo y el código generado.
El resumen y el detalle se guardan en un JSON dentro de `RESULTADOS_DIR`, que es lo que después
permite comparar las cuatro configuraciones y sacar ejemplos cualitativos.


In [ ]:
def evaluar_pass1(modelo, tokenizador, ejemplos, nombre, conjunto="test", guardar=True):
    if len(ejemplos) == 0:
        raise ValueError("No hay problemas para evaluar")

    prompts = []
    for ejemplo in ejemplos:
        prompts.append(construir_prompt(ejemplo))

    inicio_generacion = time.time()
    soluciones = generar_soluciones(modelo, tokenizador, prompts)
    segundos_generacion = time.time() - inicio_generacion

    if len(soluciones) != len(ejemplos):
        mensaje = "Se generaron " + str(len(soluciones)) + " soluciones"
        mensaje = mensaje + " para " + str(len(ejemplos)) + " problemas"
        raise ValueError(mensaje)

    inicio_ejecucion = time.time()
    resultados = []
    aciertos = 0
    for i in tqdm(range(len(ejemplos)), desc="Ejecutando", leave=False):
        codigo = extraer_codigo(soluciones[i])
        revision = ejecutar_solucion(codigo, ejemplos[i])
        if revision["pasa"]:
            aciertos = aciertos + 1
        resultados.append({
            "task_id": ejemplos[i]["task_id"],
            "pasa": revision["pasa"],
            "motivo": revision["motivo"],
            "segundos": round(revision["segundos"], 3),
            "codigo": codigo,
            "detalle": revision["detalle"],
        })
    segundos_ejecucion = time.time() - inicio_ejecucion

    motivos = {}
    for fila in resultados:
        motivo = fila["motivo"]
        if motivo in motivos:
            motivos[motivo] = motivos[motivo] + 1
        else:
            motivos[motivo] = 1

    pass1 = aciertos / len(ejemplos)
    resumen = {
        "experimento": nombre,
        "conjunto": conjunto,
        "evaluados": len(ejemplos),
        "aciertos": aciertos,
        "pass1": pass1,
        "segundos_generacion": round(segundos_generacion, 1),
        "segundos_ejecucion": round(segundos_ejecucion, 1),
        "motivos": motivos,
        "resultados": resultados,
    }

    print("Evaluación:", nombre, "(" + conjunto + ")")
    print("  problemas:", len(ejemplos))
    print("  aciertos:", aciertos)
    print("  pass@1:", round(pass1 * 100, 2), "%")
    print("  generación:", round(segundos_generacion, 1), "s")
    print("  ejecución:", round(segundos_ejecucion, 1), "s")
    print("  motivos:", motivos)

    if guardar:
        ruta = os.path.join(RESULTADOS_DIR, "pass1_" + nombre + ".json")
        with open(ruta, "w", encoding="utf-8") as archivo:
            json.dump(resumen, archivo, ensure_ascii=False, indent=2)
        print("  guardado en:", ruta)

    return resumen


Probamos el sandbox con seis soluciones escritas a mano: una correcta, una incorrecta, un bucle
infinito, un error de sintaxis, una vacía y una que depende del `test_setup_code`. La última
comprueba el orden: la solución define la clase y el setup la usa.


In [ ]:
ejemplo_suma = {
    "task_id": 1,
    "test_list": ["assert sumar(2, 3) == 5", "assert sumar(-1, 1) == 0"],
    "test_setup_code": "",
}

ejemplo_nodo = {
    "task_id": 2,
    "test_list": ["assert nodo.valor == 7"],
    "test_setup_code": "nodo = Node(7)",
}

codigo_correcto = "def sumar(a, b):\n    return a + b\n"
codigo_incorrecto = "def sumar(a, b):\n    return a - b\n"
codigo_bucle = "def sumar(a, b):\n    while True:\n        pass\n"
codigo_sintaxis = "def sumar(a, b)\n    return a + b\n"
codigo_vacio = ""
codigo_nodo = "class Node:\n    def __init__(self, valor):\n        self.valor = valor\n"

casos = [
    ("correcto", codigo_correcto, ejemplo_suma, True),
    ("incorrecto", codigo_incorrecto, ejemplo_suma, False),
    ("bucle infinito", codigo_bucle, ejemplo_suma, False),
    ("error de sintaxis", codigo_sintaxis, ejemplo_suma, False),
    ("sin código", codigo_vacio, ejemplo_suma, False),
    ("orden con setup", codigo_nodo, ejemplo_nodo, True),
]

filas_pruebas = []
for caso, codigo, ejemplo, esperado in casos:
    revision = ejecutar_solucion(codigo, ejemplo)
    filas_pruebas.append({
        "caso": caso,
        "esperado": esperado,
        "pasa": revision["pasa"],
        "motivo": revision["motivo"],
        "segundos": round(revision["segundos"], 3),
    })

tabla_pruebas = pd.DataFrame(filas_pruebas)
print(tabla_pruebas.to_string(index=False))

superadas = 0
for fila in filas_pruebas:
    if fila["pasa"] == fila["esperado"]:
        superadas = superadas + 1

print()
print("Pruebas superadas:", superadas, "de", len(filas_pruebas))


### Lo que nos deja la evaluación pass@1

**22. Un solo protocolo para las cuatro configuraciones.** `evaluar_pass1` recibe el modelo y su
tokenizador y siempre usa el mismo `construir_prompt`, el mismo `max_new_tokens`, la misma
decodificación greedy y el mismo sandbox. Cambiar de configuración solo cambia los pesos, no la
forma de evaluar. El resumen y el detalle por problema quedan en `RESULTADOS_DIR`, así que la
comparación de la sección 10 no necesita volver a generar.

**23. El sandbox aguanta código malo.** Las seis pruebas se comportaron como esperábamos: la
solución correcta pasó, la incorrecta falló por `AssertionError`, el bucle infinito se cortó con el
límite de 5 segundos, el error de sintaxis se reportó como tal, la solución vacía no llegó a
ejecutarse y la que usa el `test_setup_code` pasó solo con el orden código → setup → asserts.
Ningún caso tumbó el notebook.

**24. La extracción no arregla el código.** `extraer_codigo` quita vallas markdown, encabezados
repetidos y prosa, y conserva el código válido que empieza con una asignación, un import o una
clase. Pero si el modelo no escribe Python válido, la solución falla. Eso es correcto: el pass@1
mide lo que produce el modelo, no lo que nosotros reconstruimos.

**25. Lo que falta probar.** La generación real se estrena en la sección 7 con Qwen base. Ahí se
revisa el primer lote y se confirma que el modelo no continúa el bloque `### Signature` en vez de
empezar la solución. El sandbox ya está probado y no depende de la GPU; la generación por lotes
depende del modelo y se verifica con el primero.


## 7. Evaluación de los modelos base (zero-shot)

Los dos modelos base no han visto MBPP: se evalúan tal como salen de Hugging Face. Este es el punto
de partida contra el que se mide el aporte de LoRA y la primera vez que se usa `test`, que queda
reservado para medir y no para elegir hiperparámetros.

El protocolo es el de la sección 6: el mismo `construir_prompt`, una solución greedy por problema,
`max_new_tokens=512` y el mismo sandbox. Lo único que cambia entre las cuatro configuraciones del
proyecto son los pesos; todo lo demás se mantiene igual.


Antes de generar, comprobamos que `test` esté completo y separado de los otros dos conjuntos. Solo
miramos metadatos: tamaño, `task_id`, número de `assert` y cuántos problemas traen
`test_setup_code`.


In [ ]:
EJEMPLOS_TEST = dataset["test"]

ids_test = []
for ejemplo in EJEMPLOS_TEST:
    ids_test.append(ejemplo["task_id"])

ids_entrenamiento = []
for ejemplo in EJEMPLOS_ENTRENAMIENTO:
    ids_entrenamiento.append(ejemplo["task_id"])

ids_validacion = []
for ejemplo in EJEMPLOS_VALIDACION:
    ids_validacion.append(ejemplo["task_id"])

cantidad_asserts = []
con_setup = 0
for ejemplo in EJEMPLOS_TEST:
    cantidad_asserts.append(len(ejemplo["test_list"]))
    if ejemplo["test_setup_code"].strip() != "":
        con_setup = con_setup + 1

print("Problemas de test:", len(EJEMPLOS_TEST))
print("task_id distintos:", len(set(ids_test)))
print("task_id que también están en train:", len(set(ids_test) & set(ids_entrenamiento)))
print("task_id que también están en validation:", len(set(ids_test) & set(ids_validacion)))
print("asserts por problema:", sorted(set(cantidad_asserts)))
print("problemas con test_setup_code:", con_setup)


Cargamos Qwen2.5-1.5B base con la revisión fija y la precisión mixta del runtime. Para generar basta
con media precisión: no hay gradientes y el modelo entra holgado en la GPU. La misma función se
reutiliza en la sección 9 para cargar la base antes de pegarle el adaptador.


In [ ]:
def cargar_modelo_base(modelo_id, revision):
    modelo = AutoModelForCausalLM.from_pretrained(
        modelo_id,
        revision=revision,
        torch_dtype=AMP_DTYPE,
    )
    modelo = modelo.to(DEVICE)
    modelo.eval()
    return modelo


def contar_parametros(modelo):
    total = 0
    for parametro in modelo.parameters():
        total = total + parametro.numel()
    return total


def memoria_gpu():
    if not torch.cuda.is_available():
        return "sin GPU"
    asignada = torch.cuda.memory_allocated() / 1024 ** 3
    reservada = torch.cuda.memory_reserved() / 1024 ** 3
    texto = str(round(asignada, 2)) + " GB asignados, "
    texto = texto + str(round(reservada, 2)) + " GB reservados"
    return texto


modelo_qwen = cargar_modelo_base(QWEN_ID, QWEN_REVISION)
parametros_qwen = contar_parametros(modelo_qwen)

print("Modelo:", QWEN_ID)
print("Parámetros:", round(parametros_qwen / 1e9, 2), "mil millones")
print("VRAM:", memoria_gpu())


Antes de gastar la evaluación completa, generamos cuatro soluciones y miramos el texto crudo. Lo que
queremos confirmar es que el modelo no repita `### Signature` ni `### Problem` y que empiece a
escribir código, no una frase en prosa. No se elige nada con estas cuatro: solo se verifica el
formato del prompt.


In [ ]:
def revisar_generacion(modelo, tokenizador, ejemplos, cantidad=4):
    prompts = []
    for indice in range(cantidad):
        prompts.append(construir_prompt(ejemplos[indice]))
    textos = generar_soluciones(modelo, tokenizador, prompts, tamano_lote=cantidad)

    filas = []
    for indice in range(cantidad):
        texto = textos[indice]
        codigo = extraer_codigo(texto)
        try:
            ast.parse(codigo)
            es_valido = True
        except SyntaxError:
            es_valido = False

        primera = ""
        for linea in codigo.splitlines():
            if linea.strip() != "":
                primera = linea.strip()
                break

        empieza = primera.startswith("def ") or primera.startswith("class ")
        empieza = empieza or primera.startswith("import ") or primera.startswith("from ")
        repite = ("### Signature" in texto) or ("### Problem" in texto)
        filas.append({
            "task_id": ejemplos[indice]["task_id"],
            "repite encabezado": repite,
            "empieza con código": empieza,
            "python válido": es_valido,
            "caracteres": len(texto),
        })

        print("=" * 60)
        print("task_id:", ejemplos[indice]["task_id"])
        print(texto[:400])

    return pd.DataFrame(filas)


tabla_qwen = revisar_generacion(modelo_qwen, tokenizador_qwen, EJEMPLOS_TEST)
print()
print(tabla_qwen.to_string(index=False))


Evaluamos los 500 problemas de `test` con Qwen2.5-1.5B base. La generación greedy y el sandbox son
los mismos de la sección 6, y el JSON con el código y el motivo de cada problema queda en Drive. En
una L4 esto tarda unos minutos.


In [ ]:
resumen_qwen_base = evaluar_pass1(
    modelo_qwen,
    tokenizador_qwen,
    EJEMPLOS_TEST,
    "qwen_base",
)
print("VRAM:", memoria_gpu())


Liberamos la GPU y cargamos SmolLM2-1.7B base. La VRAM se libera poniendo el modelo en `None` y
llamando a `liberar_vram`: borrar solo el parámetro dentro de la función no suelta la referencia
del notebook.


In [ ]:
modelo_qwen = None
liberar_vram()

modelo_smol = cargar_modelo_base(SMOL_ID, SMOL_REVISION)
parametros_smol = contar_parametros(modelo_smol)

print("Modelo:", SMOL_ID)
print("Parámetros:", round(parametros_smol / 1e9, 2), "mil millones")
print("VRAM:", memoria_gpu())


Repetimos la revisión del formato con SmolLM2 sobre los mismos cuatro problemas.


In [ ]:
tabla_smol = revisar_generacion(modelo_smol, tokenizador_smol, EJEMPLOS_TEST)
print()
print(tabla_smol.to_string(index=False))


Evaluamos los 500 problemas con SmolLM2-1.7B base.


In [ ]:
resumen_smol_base = evaluar_pass1(
    modelo_smol,
    tokenizador_smol,
    EJEMPLOS_TEST,
    "smol_base",
)
print("VRAM:", memoria_gpu())


Liberamos la GPU otra vez para que la sección 8 empiece con la memoria limpia.


In [ ]:
modelo_smol = None
liberar_vram()
print("VRAM:", memoria_gpu())


Resumimos los dos resultados base. Los motivos separan los aciertos (`ok`) de los fallos por
`assert`, por sintaxis, por timeout y por código vacío, y el tiempo distingue la generación de la
ejecución en el sandbox.


In [ ]:
filas_base = []
for resumen in [resumen_qwen_base, resumen_smol_base]:
    fila = {
        "experimento": resumen["experimento"],
        "problemas": resumen["evaluados"],
        "aciertos": resumen["aciertos"],
        "pass@1 (%)": round(resumen["pass1"] * 100, 2),
        "generación (s)": resumen["segundos_generacion"],
        "ejecución (s)": resumen["segundos_ejecucion"],
    }
    for motivo in ["ok", "assert", "sintaxis", "timeout", "vacio", "error", "entorno"]:
        fila[motivo] = resumen["motivos"].get(motivo, 0)
    filas_base.append(fila)

tabla_base = pd.DataFrame(filas_base)
print(tabla_base.to_string(index=False))


## 8. Finetuning con LoRA

LoRA congela el modelo base y entrena dos matrices pequeñas en las proyecciones de atención. Así el
adaptador ocupa pocos MB, se guarda en Drive y se puede cargar sobre el mismo modelo base que ya
evaluamos.

Usamos la configuración de referencia: rango 16, alpha 16, dropout 0.05 y las proyecciones q, k, v
y o. Entrenamos 5 épocas con AdamW, learning rate 2e-4, weight decay 0, warmup corto y scheduler
coseno sobre un batch efectivo de 16. Cada época se mide con la pérdida de validación y con el
pass@1 de los 50 problemas de monitoreo; el adaptador que se guarda es el de mejor pass@1, con
desempate por menor pérdida. `test` no participa en ninguna de esas decisiones.


Fijamos los hiperparámetros y la configuración de LoRA.


In [ ]:
EPOCAS = 5
TASA_APRENDIZAJE = 2e-4
PESO_DECAIMIENTO = 0.0

configuracion_lora = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

print("Épocas:", EPOCAS)
print("Learning rate:", TASA_APRENDIZAJE)
print("Weight decay:", PESO_DECAIMIENTO)
print("Rango:", configuracion_lora.r)
print("Alpha:", configuracion_lora.lora_alpha)
print("Dropout:", configuracion_lora.lora_dropout)
print("Módulos objetivo:", configuracion_lora.target_modules)


Cargamos el modelo base en `float32` y activamos el checkpointing de gradientes: el modelo congela
sus pesos, así que la memoria se va en activaciones, y el checkpointing las recalcula en el
backward a cambio de un poco de cómputo. Con PEFT hay que pedir que la entrada requiera gradiente
y apagar la caché de generación mientras se entrena.


In [ ]:
def preparar_modelo_lora(modelo_id, revision):
    modelo = AutoModelForCausalLM.from_pretrained(
        modelo_id,
        revision=revision,
        torch_dtype=torch.float32,
    )
    modelo = modelo.to(DEVICE)
    modelo = get_peft_model(modelo, configuracion_lora)
    modelo.gradient_checkpointing_enable()
    modelo.enable_input_require_grads()
    modelo.config.use_cache = False
    return modelo


def contar_parametros_lora(modelo):
    total = 0
    for nombre, parametro in modelo.named_parameters():
        if "lora_" in nombre:
            total = total + parametro.numel()
    return total


Preparamos los lotes con `preparar_lote`, que ya deja el prompt y el relleno en `-100`. La pérdida
de validación se promedia por token supervisado y no por lote, para que los lotes con respuestas
cortas no pesen lo mismo que los largos.


In [ ]:
def mover_lote(lote):
    for clave in lote:
        lote[clave] = lote[clave].to(DEVICE)
    return lote


@torch.no_grad()
def calcular_perdida(modelo, tokenizador, ejemplos):
    modelo.eval()
    perdida_total = 0.0
    tokens_total = 0

    for inicio in range(0, len(ejemplos), BATCH_SIZE):
        bloque = ejemplos[inicio: inicio + BATCH_SIZE]
        lote = preparar_lote(bloque, tokenizador)
        lote = mover_lote(lote)
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
            salidas = modelo(**lote)
        supervisados = (lote["labels"] != -100).sum().item()
        perdida_total = perdida_total + salidas.loss.item() * supervisados
        tokens_total = tokens_total + supervisados

    return perdida_total / tokens_total


Una época recorre los 374 ejemplos en lotes de 16, en orden aleatorio con semilla, y actualiza el
optimizador y el scheduler paso a paso.


In [ ]:
def entrenar_epoca(modelo, tokenizador, ejemplos, optimizador, programador, escalador, generador):
    modelo.train()
    indices = list(range(len(ejemplos)))
    generador.shuffle(indices)
    perdida_total = 0.0
    tokens_total = 0

    for inicio in range(0, len(indices), BATCH_SIZE):
        bloque = []
        for posicion in range(inicio, min(inicio + BATCH_SIZE, len(indices))):
            bloque.append(ejemplos[indices[posicion]])
        lote = preparar_lote(bloque, tokenizador)
        lote = mover_lote(lote)

        optimizador.zero_grad()
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
            salidas = modelo(**lote)
            perdida = salidas.loss
        if USE_SCALER:
            escalador.scale(perdida).backward()
            escalador.step(optimizador)
            escalador.update()
        else:
            perdida.backward()
            optimizador.step()
        programador.step()

        supervisados = (lote["labels"] != -100).sum().item()
        perdida_total = perdida_total + perdida.item() * supervisados
        tokens_total = tokens_total + supervisados

    return perdida_total / tokens_total


El bucle completo entrena una época, mide pérdida de validación y pass@1 de monitoreo, y guarda el
adaptador si mejora. Antes de generar hay que devolverle la caché al modelo; después se apaga otra
vez para seguir entrenando. Si el adaptador ya existe en Drive, se carga y se salta el
entrenamiento, y el historial se recupera del JSON.


In [ ]:
def entrenar_lora(nombre, modelo_id, revision, tokenizador, ruta_checkpoint):
    ruta_historial = os.path.join(RESULTADOS_DIR, "historial_" + nombre + ".json")
    ruta_adaptador = os.path.join(ruta_checkpoint, "adapter_config.json")

    if os.path.exists(ruta_adaptador):
        print("Adaptador encontrado:", ruta_checkpoint)
        print("Se carga y se salta el entrenamiento.")
        modelo = cargar_modelo_base(modelo_id, revision)
        modelo = PeftModel.from_pretrained(modelo, ruta_checkpoint)
        configuracion = modelo.peft_config["default"]
        print("  r:", configuracion.r, "| alpha:", configuracion.lora_alpha)
        print("  dropout:", configuracion.lora_dropout)
        print("  módulos:", configuracion.target_modules)
        print("  parámetros LoRA:", contar_parametros_lora(modelo))
        historial = []
        if os.path.exists(ruta_historial):
            with open(ruta_historial, encoding="utf-8") as archivo:
                historial = json.load(archivo)
        modelo = None
        liberar_vram()
        return historial

    fijar_semilla()
    print("Entrenando", nombre)

    modelo = preparar_modelo_lora(modelo_id, revision)
    total = contar_parametros(modelo)
    entrenables = contar_parametros_lora(modelo)
    porcentaje = round(100 * entrenables / total, 2)
    print("Parámetros totales:", total)
    print("Parámetros LoRA:", entrenables, "(" + str(porcentaje) + "%)")

    parametros = []
    for parametro in modelo.parameters():
        if parametro.requires_grad:
            parametros.append(parametro)

    optimizador = torch.optim.AdamW(
        parametros,
        lr=TASA_APRENDIZAJE,
        weight_decay=PESO_DECAIMIENTO,
    )
    lotes_por_epoca = (len(EJEMPLOS_ENTRENAMIENTO) + BATCH_SIZE - 1) // BATCH_SIZE
    pasos_totales = lotes_por_epoca * EPOCAS
    pasos_warmup = pasos_totales // 20
    if pasos_warmup < 1:
        pasos_warmup = 1
    programador = get_cosine_schedule_with_warmup(optimizador, pasos_warmup, pasos_totales)
    escalador = torch.amp.GradScaler("cuda", enabled=USE_SCALER)
    generador = random.Random(SEED)

    print("Lotes por época:", lotes_por_epoca)
    print("Pasos totales:", pasos_totales, "| pasos de warmup:", pasos_warmup)

    historial = []
    mejor_pass1 = -1.0
    mejor_perdida = 1000000.0
    mejor_epoca = 0

    for epoca in range(1, EPOCAS + 1):
        perdida_entrenamiento = entrenar_epoca(
            modelo,
            tokenizador,
            EJEMPLOS_ENTRENAMIENTO,
            optimizador,
            programador,
            escalador,
            generador,
        )
        perdida_validacion = calcular_perdida(modelo, tokenizador, EJEMPLOS_VALIDACION)

        modelo.config.use_cache = True
        modelo.generation_config.use_cache = True
        resumen = evaluar_pass1(
            modelo,
            tokenizador,
            EJEMPLOS_MONITOREO,
            "monitoreo_" + nombre + "_epoca_" + str(epoca),
            conjunto="validation",
            guardar=False,
        )
        modelo.config.use_cache = False
        modelo.generation_config.use_cache = False
        modelo.train()

        pass1_monitoreo = resumen["pass1"]
        historial.append({
            "epoca": epoca,
            "perdida_entrenamiento": round(perdida_entrenamiento, 4),
            "perdida_validacion": round(perdida_validacion, 4),
            "pass1_monitoreo": round(pass1_monitoreo, 4),
            "mejor": False,
        })

        mejora = False
        if pass1_monitoreo > mejor_pass1:
            mejora = True
        if pass1_monitoreo == mejor_pass1 and perdida_validacion < mejor_perdida:
            mejora = True

        if mejora:
            mejor_pass1 = pass1_monitoreo
            mejor_perdida = perdida_validacion
            mejor_epoca = epoca
            for fila in historial:
                fila["mejor"] = False
            historial[-1]["mejor"] = True
            modelo.save_pretrained(ruta_checkpoint)
            tokenizador.save_pretrained(ruta_checkpoint)
            print("  nuevo mejor adaptador en la época", epoca)

        with open(ruta_historial, "w", encoding="utf-8") as archivo:
            json.dump(historial, archivo, ensure_ascii=False, indent=2)

        print("Época", epoca)
        print("  pérdida entrenamiento:", round(perdida_entrenamiento, 4))
        print("  pérdida validación:", round(perdida_validacion, 4))
        print("  pass@1 monitoreo:", round(pass1_monitoreo * 100, 2), "%")

    print()
    print("Mejor época:", mejor_epoca)
    print("Mejor pass@1 de monitoreo:", round(mejor_pass1 * 100, 2), "%")
    print("Pérdida de validación en esa época:", round(mejor_perdida, 4))
    print("Adaptador:", ruta_checkpoint)

    parametros = None
    optimizador = None
    programador = None
    escalador = None
    lote = None
    salidas = None
    modelo = None
    liberar_vram()
    return historial


Entrenamos el adaptador de Qwen2.5-1.5B. Si `checkpoint_qwen2.5_1.5b_lora` ya existe en Drive, la
celda lo carga y no entrena. En una L4 esto tarda varios minutos por época.


In [ ]:
historial_qwen = entrenar_lora(
    "qwen",
    QWEN_ID,
    QWEN_REVISION,
    tokenizador_qwen,
    CHECKPOINT_QWEN,
)
print("VRAM:", memoria_gpu())


Entrenamos el adaptador de SmolLM2-1.7B con la misma receta.


In [ ]:
historial_smol = entrenar_lora(
    "smol",
    SMOL_ID,
    SMOL_REVISION,
    tokenizador_smol,
    CHECKPOINT_SMOL,
)
print("VRAM:", memoria_gpu())


Comprobamos qué archivos quedaron en cada carpeta: el adaptador, su configuración y el tokenizador.


In [ ]:
for nombre, ruta in [("Qwen2.5-1.5B", CHECKPOINT_QWEN), ("SmolLM2-1.7B", CHECKPOINT_SMOL)]:
    print(nombre, "->", ruta)
    if os.path.exists(ruta):
        for archivo in sorted(os.listdir(ruta)):
            print("  ", archivo)
    else:
        print("   no existe")


Juntamos el historial de las dos corridas: pérdida de entrenamiento, pérdida de validación y pass@1
de monitoreo por época, con la época que quedó guardada.


In [ ]:
filas_historial = []
for nombre, historial in [("Qwen2.5-1.5B", historial_qwen), ("SmolLM2-1.7B", historial_smol)]:
    for fila in historial:
        filas_historial.append({
            "modelo": nombre,
            "época": fila["epoca"],
            "pérdida train": fila["perdida_entrenamiento"],
            "pérdida validation": fila["perdida_validacion"],
            "pass@1 monitoreo (%)": round(fila["pass1_monitoreo"] * 100, 2),
            "mejor": fila["mejor"],
        })

tabla_historial = pd.DataFrame(filas_historial)
print(tabla_historial.to_string(index=False))


Graficamos la pérdida y el pass@1 de monitoreo época por época. Si la corrida se saltó el
entrenamiento y no encontró historial, no hay curvas que mostrar.


In [ ]:
if len(tabla_historial) > 0:
    fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
    for nombre, historial in [("Qwen2.5-1.5B", historial_qwen), ("SmolLM2-1.7B", historial_smol)]:
        if len(historial) == 0:
            continue
        epocas = []
        perdidas_train = []
        perdidas_validacion = []
        pass1_monitoreo = []
        for fila in historial:
            epocas.append(fila["epoca"])
            perdidas_train.append(fila["perdida_entrenamiento"])
            perdidas_validacion.append(fila["perdida_validacion"])
            pass1_monitoreo.append(fila["pass1_monitoreo"] * 100)
        ejes[0].plot(epocas, perdidas_train, marker="o", label=nombre + " train")
        ejes[0].plot(epocas, perdidas_validacion, marker="s", linestyle="--", label=nombre + " validation")
        ejes[1].plot(epocas, pass1_monitoreo, marker="o", label=nombre)
    ejes[0].set_xlabel("época")
    ejes[0].set_ylabel("pérdida")
    ejes[0].set_title("Pérdida")
    ejes[0].legend()
    ejes[0].grid(alpha=0.3)
    ejes[1].set_xlabel("época")
    ejes[1].set_ylabel("pass@1 (%)")
    ejes[1].set_title("pass@1 en los 50 de monitoreo")
    ejes[1].legend()
    ejes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Sin historial: no hay curvas para graficar")


## 9. Evaluación de los modelos finetuneados

Cargamos cada adaptador sobre su modelo base y repetimos exactamente la evaluación de la sección 7:
los mismos 500 problemas, el mismo prompt, una solución greedy por problema y el mismo sandbox. Así
la diferencia entre base y finetuneado mide el aporte de LoRA y no un cambio de protocolo.


Antes de evaluar, comprobamos que el adaptador exista y que su configuración sea la que entrenamos:
rango 16, alpha 16, dropout 0.05 y las cuatro proyecciones de atención.


In [ ]:
def cargar_modelo_con_adapter(modelo_id, revision, ruta_checkpoint):
    ruta_adaptador = os.path.join(ruta_checkpoint, "adapter_config.json")
    if not os.path.exists(ruta_adaptador):
        raise FileNotFoundError("No existe el adaptador en " + ruta_checkpoint)
    modelo = cargar_modelo_base(modelo_id, revision)
    modelo = PeftModel.from_pretrained(modelo, ruta_checkpoint)
    modelo.eval()
    return modelo


modelo_qwen_lora = cargar_modelo_con_adapter(QWEN_ID, QWEN_REVISION, CHECKPOINT_QWEN)
configuracion = modelo_qwen_lora.peft_config["default"]

print("Modelo:", QWEN_ID)
print("  r:", configuracion.r)
print("  alpha:", configuracion.lora_alpha)
print("  dropout:", configuracion.lora_dropout)
print("  módulos:", configuracion.target_modules)
print("  parámetros LoRA:", contar_parametros_lora(modelo_qwen_lora))
print("VRAM:", memoria_gpu())


Evaluamos los 500 problemas con Qwen2.5-1.5B + LoRA.


In [ ]:
resumen_qwen_lora = evaluar_pass1(
    modelo_qwen_lora,
    tokenizador_qwen,
    EJEMPLOS_TEST,
    "qwen_lora",
)
print("VRAM:", memoria_gpu())


Liberamos la GPU y cargamos el adaptador de SmolLM2 sobre su base.


In [ ]:
modelo_qwen_lora = None
liberar_vram()

modelo_smol_lora = cargar_modelo_con_adapter(SMOL_ID, SMOL_REVISION, CHECKPOINT_SMOL)
configuracion = modelo_smol_lora.peft_config["default"]

print("Modelo:", SMOL_ID)
print("  r:", configuracion.r)
print("  alpha:", configuracion.lora_alpha)
print("  dropout:", configuracion.lora_dropout)
print("  módulos:", configuracion.target_modules)
print("  parámetros LoRA:", contar_parametros_lora(modelo_smol_lora))
print("VRAM:", memoria_gpu())


Evaluamos los 500 problemas con SmolLM2-1.7B + LoRA.


In [ ]:
resumen_smol_lora = evaluar_pass1(
    modelo_smol_lora,
    tokenizador_smol,
    EJEMPLOS_TEST,
    "smol_lora",
)
print("VRAM:", memoria_gpu())


Liberamos la GPU al terminar.


In [ ]:
modelo_smol_lora = None
liberar_vram()
print("VRAM:", memoria_gpu())


Resumimos los dos resultados finetuneados con el mismo formato de la tabla base.


In [ ]:
filas_lora = []
for resumen in [resumen_qwen_lora, resumen_smol_lora]:
    fila = {
        "experimento": resumen["experimento"],
        "problemas": resumen["evaluados"],
        "aciertos": resumen["aciertos"],
        "pass@1 (%)": round(resumen["pass1"] * 100, 2),
        "generación (s)": resumen["segundos_generacion"],
        "ejecución (s)": resumen["segundos_ejecucion"],
    }
    for motivo in ["ok", "assert", "sintaxis", "timeout", "vacio", "error", "entorno"]:
        fila[motivo] = resumen["motivos"].get(motivo, 0)
    filas_lora.append(fila)

tabla_lora = pd.DataFrame(filas_lora)
print(tabla_lora.to_string(index=False))


## 10. Comparación y resultados

Pendiente: tabla de las cuatro configuraciones, gráficas y ejemplos cualitativos.


## 11. Conclusiones

Pendiente: discusión de resultados, seguimiento del entrenamiento y limitaciones.
